# Skytrax raw EDA

**Convention:** EDA only reads raw tables and never modifies the data. Cleaning rules are written later in dbt staging, after we understand the data.

**Standard process for every table** (every table section below has exactly these headings, in this order):
1. **Shape & schema**: row count, column names, DuckDB-inferred types, 10 sample rows.
2. **Column descriptions**: what each column is about (one line each, written in markdown).
3. **Nulls & coverage**: null count and null % for every column.
4. **Profile by column type**:
   - 4a. varchar: count(distinct), min/max length, most frequent value, top values for categorical columns.
   - 4b. numeric: min, max, avg, median, histogram.
   - 4c. date: min, max, count of anomalous dates (before 1990, future dates).
   - 4d. boolean: distribution of True / False / null.
5. **Caveat hunting**: keys and duplicates, orphaned foreign keys, whitespace, single-character values, upper/lower case inconsistencies, invalid dates, values shifted into the wrong column.
6. **Summary**: purpose, shape, column table, caveats with planned handling in staging. (Later copied to `docs/` as the data dictionary.)

**How to work through a table:** run the cells top to bottom. Steps 3, 4 and parts of 5 call the helper functions defined in the setup cell below, so every table is profiled with the same code. A sub-step that does not apply (for example 4c when a table has no date column) is still kept, and its helper prints a "no columns" message.

**Tables:**

| # | Table | Role | 
|---|---|---|
| 1 | `raw_airlines` | dimension | 
| 2 | `raw_airports` | dimension |
| 3 | `raw_seat_reviews` | fact | 
| 4 | `raw_lounge_reviews` | fact | 
| 5 | `raw_airport_reviews` | fact | 
| 6 | `raw_airline_reviews` | fact |

## Setup and helper functions

Run this cell first. Every helper takes a table name and reads the column names and types from `DESCRIBE`, so the same call works for any table.

| Helper | Used in | What it shows |
|---|---|---|
| `get_columns(table, *types)` | building block | column names of a table, optionally only some DuckDB types |
| `show_union(queries)` | building block | runs one small query per column and prints them as one table |
| `null_check(table)` | step 3 | null count and null % for every column |
| `profile_overview(table)` | step 4 | one line per column: type, distinct count, min and max (length for varchar) |
| `varchar_profile(table)` | 4a | distinct count, shortest/longest text, most frequent value, per varchar column |
| `top_values(table, col, n)` | 4a | the n most frequent values of one column |
| `numeric_stats(table)` | 4b | distinct count, min, max, average, median per BIGINT/DOUBLE column |
| `numeric_hist(table)` | 4b | how many rows have each value, per rating (DOUBLE) column |
| `date_stats(table)` | 4c | min, max, dates before 1990, future dates, per DATE/TIMESTAMP column |
| `bool_dist(table)` | 4d | True / False / null counts per BOOLEAN column |
| `key_checks(table, key, parents)` | 5 | duplicate keys, duplicate full rows, orphaned foreign keys |
| `whitespace_check(table)` | 5 | values with leading/trailing spaces and 1-character values, per varchar column |
| `dup_text(table, col)` | 5 | rows whose text also appears in another row |

In [1]:
import duckdb

# ---------------------------------------------------------------------------
# Connection
# read_only=True means EDA can never change the raw data by accident.
# ---------------------------------------------------------------------------
con = duckdb.connect("../database/dev.duckdb", read_only=True)

# DuckDB type names, grouped so helpers can pick the columns they need.
NUMERIC_TYPES = ("BIGINT", "DOUBLE")
DATE_TYPES = ("DATE", "TIMESTAMP")


# ---------------------------------------------------------------------------
# Building blocks used by the other helpers
# ---------------------------------------------------------------------------
def get_columns(table, *types):
    """Return the column names of a table.

    get_columns("raw_airlines")             -> ["airline_id", "airline_name"]
    get_columns("raw_airlines", "VARCHAR")  -> ["airline_name"]  (only varchar columns)
    """
    names = []
    for name, col_type, *_ in con.sql(f"DESCRIBE {table}").fetchall():
        # No type filter given -> keep every column. Otherwise keep matching types only.
        if len(types) == 0 or col_type in types:
            names.append(name)
    return names


def show_union(queries, order_by=None, max_rows=100):
    """Run several one-row SELECT queries as one query and print the result.

    Most helpers build one small query per column. UNION ALL stacks those rows
    into a single table, so the output has one line per column.
    """
    sql = " UNION ALL ".join(queries)
    if order_by:
        sql = sql + f" ORDER BY {order_by}"
    con.sql(sql).show(max_rows=max_rows, max_width=200)


# ---------------------------------------------------------------------------
# Step 3: nulls and coverage
# ---------------------------------------------------------------------------
def null_check(table):
    """Step 3. For every column: how many values are NULL, and what % of rows.

    Example output row:  col = date_visit | n_null = 2290 | pct_null = 45.02
    COUNT(*) counts all rows, COUNT(col) counts only non-null values,
    so the difference is the number of nulls.
    """
    queries = []
    for col in get_columns(table):
        query = f"""
            SELECT
                '{col}'                                                        AS col,
                COUNT(*) - COUNT("{col}")                                      AS n_null,
                ROUND(100.0 * (COUNT(*) - COUNT("{col}")) / COUNT(*), 2)       AS pct_null
            FROM {table}
        """
        queries.append(query)
    show_union(queries, order_by="pct_null DESC, col")   # col breaks ties so the order is stable


# ---------------------------------------------------------------------------
# Step 4: profile by column type
# ---------------------------------------------------------------------------
def profile_overview(table):
    """Step 4 overview. One line per column: type, distinct count, min and max.

    For varchar columns min/max are the shortest/longest text LENGTH.
    For numeric and date columns min/max are the smallest/largest value.
    For boolean columns min/max are not meaningful, so they stay NULL.
    """
    queries = []
    for col, col_type, *_ in con.sql(f"DESCRIBE {table}").fetchall():
        if col_type == "VARCHAR":
            min_expr = f'MIN(LENGTH("{col}"))::VARCHAR'
            max_expr = f'MAX(LENGTH("{col}"))::VARCHAR'
        elif col_type in NUMERIC_TYPES + DATE_TYPES:
            min_expr = f'MIN("{col}")::VARCHAR'
            max_expr = f'MAX("{col}")::VARCHAR'
        else:
            min_expr = "NULL"
            max_expr = "NULL"

        query = f"""
            SELECT
                '{col}'                   AS col,
                '{col_type}'              AS col_type,
                COUNT(DISTINCT "{col}")   AS n_distinct,
                {min_expr}                AS min_val,
                {max_expr}                AS max_val
            FROM {table}
        """
        queries.append(query)
    show_union(queries)


def varchar_profile(table):
    """Step 4a. For every varchar column: distinct count, shortest and longest
    text, and the most frequent value (cut to 40 characters so it fits).
    """
    columns = get_columns(table, "VARCHAR")
    if len(columns) == 0:
        print(f"No varchar columns in {table}")
        return

    queries = []
    for col in columns:
        query = f"""
            SELECT
                '{col}'                          AS col,
                COUNT(DISTINCT "{col}")          AS n_distinct,
                MIN(LENGTH("{col}"))             AS min_len,
                MAX(LENGTH("{col}"))             AS max_len,
                LEFT(MODE("{col}"), 40)          AS most_frequent
            FROM {table}
        """
        queries.append(query)
    show_union(queries)


def top_values(table, col, n=10):
    """Step 4a. The n most frequent values of ONE column, with count and %.

    Use it on categorical columns (type_of_traveller, seat_type, ...)
    to see every spelling that exists, including typos and case variants.
    """
    sql = f"""
        SELECT
            "{col}"                                                       AS value,
            COUNT(*)                                                      AS n,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)            AS pct
        FROM {table}
        GROUP BY "{col}"
        ORDER BY n DESC, value
        LIMIT {n}
    """
    con.sql(sql).show(max_rows=n, max_width=200)


def numeric_stats(table):
    """Step 4b. For every numeric column (BIGINT, DOUBLE): distinct count,
    min, max, average and median.
    """
    columns = get_columns(table, *NUMERIC_TYPES)
    if len(columns) == 0:
        print(f"No numeric columns in {table}")
        return

    queries = []
    for col in columns:
        query = f"""
            SELECT
                '{col}'                      AS col,
                COUNT(DISTINCT "{col}")      AS n_distinct,
                MIN("{col}")                 AS min_val,
                MAX("{col}")                 AS max_val,
                ROUND(AVG("{col}"), 2)       AS avg_val,
                MEDIAN("{col}")              AS median_val
            FROM {table}
        """
        queries.append(query)
    show_union(queries)


def numeric_hist(table):
    """Step 4b. Histogram of every rating column (DOUBLE type).

    Output has one row per column and one column per rating value, e.g.
        col       |   1.0 |  2.0 |  3.0 ...
        comfort   |   900 |  400 |  700 ...
    Ratings should only be whole numbers 1 to 5. Anything else is a problem.
    """
    columns = get_columns(table, "DOUBLE")
    if len(columns) == 0:
        print(f"No rating (DOUBLE) columns in {table}")
        return

    # Step 1: for each column, count how many rows have each value.
    #         Result: (col, value, n)
    counting_queries = []
    for col in columns:
        query = f"""
            SELECT '{col}' AS col, "{col}" AS value, COUNT(*) AS n
            FROM {table}
            WHERE "{col}" IS NOT NULL
            GROUP BY "{col}"
        """
        counting_queries.append(query)
    counts = " UNION ALL ".join(counting_queries)

    # Step 2: PIVOT turns each distinct value into its own output column.
    con.sql(f"PIVOT ({counts}) ON value USING SUM(n) GROUP BY col ORDER BY col").show(max_rows=100, max_width=200)


def date_stats(table):
    """Step 4c. For every DATE / TIMESTAMP column: earliest, latest,
    how many dates are before 1990 (placeholders like 1970-01-01),
    how many are in the future, and the distinct count.
    """
    columns = get_columns(table, *DATE_TYPES)
    if len(columns) == 0:
        print(f"No date columns in {table}")
        return

    queries = []
    for col in columns:
        query = f"""
            SELECT
                '{col}'                                                  AS col,
                MIN("{col}")::VARCHAR                                    AS min_val,
                MAX("{col}")::VARCHAR                                    AS max_val,
                COUNT(*) FILTER (WHERE "{col}" < '1990-01-01')           AS n_before_1990,
                COUNT(*) FILTER (WHERE "{col}" > current_date)           AS n_future,
                COUNT(DISTINCT "{col}")                                  AS n_distinct
            FROM {table}
        """
        queries.append(query)
    show_union(queries)


def bool_dist(table):
    """Step 4d. For every BOOLEAN column: how many True, False and NULL."""
    columns = get_columns(table, "BOOLEAN")
    if len(columns) == 0:
        print(f"No boolean columns in {table}")
        return

    queries = []
    for col in columns:
        query = f"""
            SELECT
                '{col}'                                          AS col,
                COUNT(*) FILTER (WHERE "{col}")                  AS n_true,
                COUNT(*) FILTER (WHERE NOT "{col}")              AS n_false,
                COUNT(*) FILTER (WHERE "{col}" IS NULL)          AS n_null
            FROM {table}
        """
        queries.append(query)
    show_union(queries)


# ---------------------------------------------------------------------------
# Step 5: caveat hunting
# ---------------------------------------------------------------------------
def key_checks(table, key, parents=None):
    """Step 5. Check the primary key and the foreign keys of a table.

    key      the column that should be unique, e.g. "review_id"
    parents  optional dict {foreign_key_column: parent_table}, e.g.
             {"airline_id": "raw_airlines"}

    Output columns:
      duplicate_keys       rows minus distinct keys (0 = key is unique)
      duplicate_full_rows  rows that are exact copies of another row
      orphan_<fk>          rows whose foreign key has no match in the parent
                           table (0 = every review points to a real airline)
    """
    # Each item below becomes one column of the single output row.
    parts = []

    parts.append(f"(SELECT COUNT(*) - COUNT(DISTINCT {key}) FROM {table}) AS duplicate_keys")

    distinct_rows = f"(SELECT DISTINCT * FROM {table})"
    parts.append(f"(SELECT COUNT(*) FROM {table}) - (SELECT COUNT(*) FROM {distinct_rows}) AS duplicate_full_rows")

    if parents is not None:
        for fk_column, parent_table in parents.items():
            # LEFT JOIN keeps every review. If the parent has no match,
            # the parent's column is NULL, and that row is an orphan.
            orphan_query = f"""
                (SELECT COUNT(*)
                 FROM {table} AS child
                 LEFT JOIN {parent_table} AS parent USING ({fk_column})
                 WHERE parent.{fk_column} IS NULL) AS orphan_{fk_column}
            """
            parts.append(orphan_query)

    con.sql("SELECT " + ", ".join(parts)).show(max_width=200)


def whitespace_check(table):
    """Step 5. For every varchar column count:
      n_whitespace  values with a space at the start or end (' Qatar ')
      n_one_char    values that are a single character ('=')
    Both are usually scraping leftovers that staging should clean.
    """
    queries = []
    for col in get_columns(table, "VARCHAR"):
        query = f"""
            SELECT
                '{col}'                                               AS col,
                COUNT(*) FILTER (WHERE "{col}" <> TRIM("{col}"))      AS n_whitespace,
                COUNT(*) FILTER (WHERE LENGTH("{col}") = 1)           AS n_one_char
            FROM {table}
        """
        queries.append(query)
    show_union(queries)


def dup_text(table, col):
    """Step 5. Count rows whose text in `col` also appears in another row.

    Example: dup_text("raw_seat_reviews", "review") -> 672 means 672 rows
    share their review text with at least one other row.
    """
    sql = f"""
        SELECT COUNT(*) AS rows_with_duplicate_{col}
        FROM {table}
        WHERE "{col}" IN (
            SELECT "{col}"
            FROM {table}
            GROUP BY "{col}"
            HAVING COUNT(*) > 1
        )
    """
    con.sql(sql).show(max_width=200)

# Table 1: `raw_airlines`

**Role:** Dimension table of airlines. Join key `airline_id` is used by every review table.

**Steps:** 1 Shape & schema, 2 Column descriptions, 3 Nulls & coverage, 4 Profile by type (4a varchar, 4b numeric, 4c date, 4d boolean), 5 Caveat hunting, 6 Summary.

In [2]:
T = "raw_airlines"

## Step 1: Shape & schema

How many rows and columns, what types DuckDB inferred, and what the data looks like.

In [3]:
con.sql(f"describe {T}").show(max_rows=40, max_width=200)

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ airline_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ airline_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [4]:
con.sql(f"select count(*) as n_rows from {T}")

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│    595 │
└────────┘

In [5]:
con.sql(f"select * from {T} limit 10").show(max_width=200)

┌────────────┬────────────────────────┐
│ airline_id │      airline_name      │
│   int64    │        varchar         │
├────────────┼────────────────────────┤
│          1 │ AB Aviation            │
│          2 │ AIRDO                  │
│          3 │ AJet                   │
│          4 │ ANA All Nippon Airways │
│          5 │ ASKY Airlines          │
│          6 │ ATA Airlines           │
│          7 │ Adria Airways          │
│          8 │ Aegean Airlines        │
│          9 │ Aer Lingus             │
│         10 │ Aero VIP               │
└────────────┴────────────────────────┘
  10 rows                   2 columns



## Step 2: Column descriptions

One line per column. Meanings are inferred from the column name and the data, so correct them if you know better.

| Column | Type | Meaning |
|---|---|---|
| `airline_id` | BIGINT | Numeric identifier of the airline |
| `airline_name` | VARCHAR | Display name of the airline |

## Step 3: Nulls & coverage

Null count and null % for every column, highest first. A high null % is real missing data, not something to fill.

In [6]:
null_check(T)

┌──────────────┬────────┬──────────┐
│     col      │ n_null │ pct_null │
│   varchar    │ int64  │  double  │
├──────────────┼────────┼──────────┤
│ airline_id   │      0 │      0.0 │
│ airline_name │      0 │      0.0 │
└──────────────┴────────┴──────────┘



## Step 4: Profile by column type

Overview first (one line per column), then the type-specific checks.

In [7]:
profile_overview(T)

┌──────────────┬──────────┬────────────┬─────────┬─────────┐
│     col      │ col_type │ n_distinct │ min_val │ max_val │
│   varchar    │ varchar  │   int64    │ varchar │ varchar │
├──────────────┼──────────┼────────────┼─────────┼─────────┤
│ airline_id   │ BIGINT   │        595 │ 1       │ 595     │
│ airline_name │ VARCHAR  │        595 │ 4       │ 32      │
└──────────────┴──────────┴────────────┴─────────┴─────────┘



### 4a. varchar

Distinct count, min/max length, most frequent value. Then the top values of categorical columns.

In [8]:
varchar_profile(T)

┌──────────────┬────────────┬─────────┬─────────┬───────────────┐
│     col      │ n_distinct │ min_len │ max_len │ most_frequent │
│   varchar    │   int64    │  int64  │  int64  │    varchar    │
├──────────────┼────────────┼─────────┼─────────┼───────────────┤
│ airline_name │        595 │       4 │      32 │ AB Aviation   │
└──────────────┴────────────┴─────────┴─────────┴───────────────┘



### 4b. numeric

Min, max, avg, median. For rating columns the histogram should show whole numbers 1 to 5 only.

In [9]:
numeric_stats(T)

┌────────────┬────────────┬─────────┬─────────┬─────────┬────────────┐
│    col     │ n_distinct │ min_val │ max_val │ avg_val │ median_val │
│  varchar   │   int64    │  int64  │  int64  │ double  │   double   │
├────────────┼────────────┼─────────┼─────────┼─────────┼────────────┤
│ airline_id │        595 │       1 │     595 │   298.0 │      298.0 │
└────────────┴────────────┴─────────┴─────────┴─────────┴────────────┘



In [10]:
numeric_hist(T)

No rating (DOUBLE) columns in raw_airlines


### 4c. date

Min, max, rows before 1990 (placeholder dates such as 1970-01-01) and future dates.

In [11]:
date_stats(T)

No date columns in raw_airlines


### 4d. boolean

Distribution of True / False / null.

In [12]:
bool_dist(T)

No boolean columns in raw_airlines


## Step 5: Caveat hunting

Keys and duplicates first, then whitespace, then table-specific checks. Anything found here goes in the summary with a planned fix for dbt staging.

In [13]:
key_checks(T, 'airline_id')

┌────────────────┬─────────────────────┐
│ duplicate_keys │ duplicate_full_rows │
│     int64      │        int64        │
├────────────────┼─────────────────────┤
│              0 │                   0 │
└────────────────┴─────────────────────┘



In [14]:
whitespace_check(T)

┌──────────────┬──────────────┬────────────┐
│     col      │ n_whitespace │ n_one_char │
│   varchar    │    int64     │   int64    │
├──────────────┼──────────────┼────────────┤
│ airline_name │            0 │          0 │
└──────────────┴──────────────┴────────────┘



In [15]:
con.sql(f"select * from {T} where length(airline_name) = 1")

┌────────────┬──────────────┐
│ airline_id │ airline_name │
│   int64    │   varchar    │
└────────────┴──────────────┘
           0 rows          

In [16]:
# same name written with different case
con.sql(f"""
    select lower(airline_name) name_lower, count(*) duplicates
    from {T} group by 1 having count(*) > 1
""")

┌────────────┬────────────┐
│ name_lower │ duplicates │
│  varchar   │   int64    │
└────────────┴────────────┘
          0 rows         

In [17]:
# id gaps: 0 means ids run 1..N with no hole
con.sql(f"select max(airline_id) - min(airline_id) + 1 - count(*) as n_gaps from {T}")

┌────────┐
│ n_gaps │
│ int64  │
├────────┤
│      0 │
└────────┘

## Step 6: Summary of `raw_airlines`

**Purpose:** Dimension table of airlines.

**Shape:** 595 rows x 2 columns.

| Column | DuckDB type | Meaning | Null | Distinct | Profile |
|---|---|---|---|---|---|
| `airline_id` | BIGINT | Numeric identifier of the airline | 0 (0%) | 595 | min 1, max 595, no gaps (595 rows = 595 distinct = max id) |
| `airline_name` | VARCHAR | Display name of the airline | 0 (0%) | 595 | length 4-32 characters |

**Caveat checks (all clean):**
- Null values: none in either column.
- Duplicate `airline_id`: 0.
- Duplicate `airline_name`, case-insensitive: 0.
- Leading/trailing whitespace in `airline_name`: 0.
- Single-character names: 0.

**Conclusion:** `raw_airlines` is clean. `airline_id` is a unique, non-null integer key, so it can be treated as a primary key.

# Table 2: `raw_airports`

**Role:** Dimension table of airports. `airport_id` is the join key used by `raw_airport_reviews`.

**Steps:** 1 Shape & schema, 2 Column descriptions, 3 Nulls & coverage, 4 Profile by type (4a varchar, 4b numeric, 4c date, 4d boolean), 5 Caveat hunting, 6 Summary.

In [18]:
T = "raw_airports"

## Step 1: Shape & schema

How many rows and columns, what types DuckDB inferred, and what the data looks like.

In [19]:
con.sql(f"describe {T}").show(max_rows=40, max_width=200)

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ airport_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ airport_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [20]:
con.sql(f"select count(*) as n_rows from {T}")

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   1004 │
└────────┘

In [21]:
con.sql(f"select * from {T} limit 10").show(max_width=200)

┌────────────┬────────────────────────────┐
│ airport_id │        airport_name        │
│   int64    │          varchar           │
├────────────┼────────────────────────────┤
│          1 │ ANR Robinson International │
│          2 │ Aalborg                    │
│          3 │ Aarhus                     │
│          4 │ Abbotsford Intl            │
│          5 │ Aberdeen                   │
│          6 │ Abidjan                    │
│          7 │ Abu Dhabi                  │
│          8 │ Abuja                      │
│          9 │ Acapulco                   │
│         10 │ Accra Kotoka               │
└────────────┴────────────────────────────┘
  10 rows                       2 columns



## Step 2: Column descriptions

One line per column. Meanings are inferred from the column name and the data, so correct them if you know better.

| Column | Type | Meaning |
|---|---|---|
| `airport_id` | BIGINT | Numeric identifier of the airport |
| `airport_name` | VARCHAR | Display name of the airport |

## Step 3: Nulls & coverage

Null count and null % for every column, highest first. A high null % is real missing data, not something to fill.

In [22]:
null_check(T)

┌──────────────┬────────┬──────────┐
│     col      │ n_null │ pct_null │
│   varchar    │ int64  │  double  │
├──────────────┼────────┼──────────┤
│ airport_id   │      0 │      0.0 │
│ airport_name │      0 │      0.0 │
└──────────────┴────────┴──────────┘



## Step 4: Profile by column type

Overview first (one line per column), then the type-specific checks.

In [23]:
profile_overview(T)

┌──────────────┬──────────┬────────────┬─────────┬─────────┐
│     col      │ col_type │ n_distinct │ min_val │ max_val │
│   varchar    │ varchar  │   int64    │ varchar │ varchar │
├──────────────┼──────────┼────────────┼─────────┼─────────┤
│ airport_id   │ BIGINT   │       1004 │ 1       │ 1004    │
│ airport_name │ VARCHAR  │       1004 │ 3       │ 30      │
└──────────────┴──────────┴────────────┴─────────┴─────────┘



### 4a. varchar

Distinct count, min/max length, most frequent value. Then the top values of categorical columns.

In [24]:
varchar_profile(T)

┌──────────────┬────────────┬─────────┬─────────┬────────────────────────────┐
│     col      │ n_distinct │ min_len │ max_len │       most_frequent        │
│   varchar    │   int64    │  int64  │  int64  │          varchar           │
├──────────────┼────────────┼─────────┼─────────┼────────────────────────────┤
│ airport_name │       1004 │       3 │      30 │ ANR Robinson International │
└──────────────┴────────────┴─────────┴─────────┴────────────────────────────┘



### 4b. numeric

Min, max, avg, median. For rating columns the histogram should show whole numbers 1 to 5 only.

In [25]:
numeric_stats(T)

┌────────────┬────────────┬─────────┬─────────┬─────────┬────────────┐
│    col     │ n_distinct │ min_val │ max_val │ avg_val │ median_val │
│  varchar   │   int64    │  int64  │  int64  │ double  │   double   │
├────────────┼────────────┼─────────┼─────────┼─────────┼────────────┤
│ airport_id │       1004 │       1 │    1004 │   502.5 │      502.5 │
└────────────┴────────────┴─────────┴─────────┴─────────┴────────────┘



In [26]:
numeric_hist(T)

No rating (DOUBLE) columns in raw_airports


### 4c. date

Min, max, rows before 1990 (placeholder dates such as 1970-01-01) and future dates.

In [27]:
date_stats(T)

No date columns in raw_airports


### 4d. boolean

Distribution of True / False / null.

In [28]:
bool_dist(T)

No boolean columns in raw_airports


## Step 5: Caveat hunting

Keys and duplicates first, then whitespace, then table-specific checks. Anything found here goes in the summary with a planned fix for dbt staging.

In [29]:
key_checks(T, 'airport_id')

┌────────────────┬─────────────────────┐
│ duplicate_keys │ duplicate_full_rows │
│     int64      │        int64        │
├────────────────┼─────────────────────┤
│              0 │                   0 │
└────────────────┴─────────────────────┘



In [30]:
whitespace_check(T)

┌──────────────┬──────────────┬────────────┐
│     col      │ n_whitespace │ n_one_char │
│   varchar    │    int64     │   int64    │
├──────────────┼──────────────┼────────────┤
│ airport_name │            0 │          0 │
└──────────────┴──────────────┴────────────┘



In [31]:
con.sql(f"select * from {T} where length(airport_name) <= 3")

┌────────────┬──────────────┐
│ airport_id │ airport_name │
│   int64    │   varchar    │
├────────────┼──────────────┤
│        334 │ Goa          │
│        480 │ Kos          │
│        508 │ Leh          │
│        694 │ Pau          │
│        786 │ Sal          │
└────────────┴──────────────┘

In [32]:
# same name written with different case
con.sql(f"""
    select lower(airport_name) name_lower, count(*) duplicates
    from {T} group by 1 having count(*) > 1
""")

┌────────────┬────────────┐
│ name_lower │ duplicates │
│  varchar   │   int64    │
└────────────┴────────────┘
          0 rows         

In [33]:
# non-ASCII characters: accents and en-dash vs plain hyphen
con.sql(f"select * from {T} where airport_name <> strip_accents(airport_name) or airport_name like '%' || chr(8211) || '%'")

┌────────────┬────────────────────────────┐
│ airport_id │        airport_name        │
│   int64    │          varchar           │
├────────────┼────────────────────────────┤
│         21 │ Akron–Canton               │
│         62 │ Bacolod–Silay              │
│        236 │ Cúcuta                     │
│        271 │ Douglas–Charles            │
│        306 │ Fes-Saïss                  │
│        346 │ Greenville–Spartanburg     │
│        414 │ Istanbul Sabiha Gökçen     │
│        539 │ Luleå                      │
│        563 │ Manchester–Boston Regional │
│        607 │ Montréal                   │
│        646 │ Norrköping                 │
│        665 │ Ostend–Bruges              │
│        715 │ Pointe-à-Pitre             │
│        782 │ Rzeszów-Jasionka           │
│        783 │ Río Grande                 │
│        881 │ São Paulo Viracopos        │
└────────────┴────────────────────────────┘
  16 rows                       2 columns

In [34]:
# orphaned foreign keys from the child table: reviews whose airport_id is missing here
con.sql("""
    select count(*) as orphan_airport_reviews
    from raw_airport_reviews r left join raw_airports a using (airport_id)
    where a.airport_id is null
""")

┌────────────────────────┐
│ orphan_airport_reviews │
│         int64          │
├────────────────────────┤
│                      0 │
└────────────────────────┘

In [35]:
# id gaps: 0 means ids run 1..N with no hole
con.sql(f"select max(airport_id) - min(airport_id) + 1 - count(*) as n_gaps from {T}")

┌────────┐
│ n_gaps │
│ int64  │
├────────┤
│      0 │
└────────┘

## Step 6: Summary of `raw_airports`

**Purpose:** Dimension table of airports. `airport_id` is the join key used by `raw_airport_reviews`.

**Shape:** 1,004 rows x 2 columns.

| Column | DuckDB type | Meaning | Null | Distinct | Profile |
|---|---|---|---|---|---|
| `airport_id` | BIGINT | Numeric identifier of the airport | 0 (0%) | 1,004 | min 1, max 1004, no gaps |
| `airport_name` | VARCHAR | Display name of the airport | 0 (0%) | 1,004 | length 3-30 characters |

**Caveat checks:**
- Null values: none in either column.
- Duplicate `airport_id`: 0.
- Duplicate `airport_name`, case-insensitive: 0.
- Leading/trailing whitespace: 0.
- Orphaned foreign keys: 0 (every `airport_id` in `raw_airport_reviews` exists here).
- **Non-ASCII characters (about 16 rows).** Accented letters (e.g. Cucuta, Lulea, Montreal, Sao Paulo Viracopos, written with accents in the data) and en-dashes (e.g. Akron-Canton, Manchester-Boston Regional, written with an en-dash in the data). The same dataset also uses a plain hyphen (e.g. Fes-Saiss), so dash style is inconsistent.
- Short names (3 characters): Goa, Kos, Leh, Pau, Sal. These are real airport names, not errors.

**Conclusion:** `raw_airports` is clean and safe to use as a dimension

# Table 3: `raw_seat_reviews`

**Role:** Fact table of seat reviews, one row per review. Joins to `raw_airlines` on `airline_id`.

**Steps:** 1 Shape & schema, 2 Column descriptions, 3 Nulls & coverage, 4 Profile by type (4a varchar, 4b numeric, 4c date, 4d boolean), 5 Caveat hunting, 6 Summary.

In [36]:
T = "raw_seat_reviews"

## Step 1: Shape & schema

How many rows and columns, what types DuckDB inferred, and what the data looks like.

In [37]:
con.sql(f"describe {T}").show(max_rows=40, max_width=200)

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ review_id         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ airline_id        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ type_of_traveller │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ seat_type         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ aircraft_type     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ seat_layout       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ verify            │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_submitted    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ date_flown        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_n

In [38]:
con.sql(f"select count(*) as n_rows from {T}")

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   3766 │
└────────┘

In [39]:
con.sql(f"select * from {T} limit 10").show(max_width=200)

┌───────────┬────────────┬───────────────────┬─────────────────┬────────────────┬─────────────┬───┬─────────────────┬──────────────┬─────────────┬───────────────────────────────┬─────────────────────┐
│ review_id │ airline_id │ type_of_traveller │    seat_type    │ aircraft_type  │ seat_layout │ … │ seat_bed_length │ seat_privacy │ recommended │            review             │     updated_at      │
│   int64   │   int64    │      varchar      │     varchar     │    varchar     │   varchar   │ … │     double      │    double    │   boolean   │            varchar            │      timestamp      │
├───────────┼────────────┼───────────────────┼─────────────────┼────────────────┼─────────────┼───┼─────────────────┼──────────────┼─────────────┼───────────────────────────────┼─────────────────────┤
│         1 │         17 │ Couple Leisure    │ Economy Class   │ Boeing 737-800 │ 3x3         │ … │            NULL │         NULL │ false       │ The seats on the way to Canc… │ 2026-06-20 12:45:

## Step 2: Column descriptions

One line per column. Meanings are inferred from the column name and the data, so correct them if you know better.

| Column | Type | Meaning |
|---|---|---|
| `review_id` | BIGINT | Review id (1 to 3766) |
| `airline_id` | BIGINT | Airline reviewed, joins to `raw_airlines` |
| `type_of_traveller` | VARCHAR | Trip type (Solo Leisure, Couple Leisure, Family Leisure, Business, ...) |
| `seat_type` | VARCHAR | Cabin class (Economy Class, Business Class, ...) |
| `aircraft_type` | VARCHAR | Aircraft model flown |
| `seat_layout` | VARCHAR | Seat configuration such as 3x3x3 |
| `verify` | BOOLEAN | Verified review |
| `date_submitted` | DATE | Date the review was posted |
| `date_flown` | DATE | Month of the flight (first day of month) |
| `customer_name` | VARCHAR | Reviewer name |
| `nationality` | VARCHAR | Reviewer country |
| `seat_legroom`, `seat_recline`, `seat_width`, `aisle_space` | DOUBLE | Core ratings 1 to 5 |
| `seat_storage`, `power_supply`, `viewing_tv_screen` | DOUBLE | Optional ratings 1 to 5 |
| `sleep_comfort`, `sitting_comfort`, `seat_bed_width`, `seat_bed_length`, `seat_privacy` | DOUBLE | Premium (lie-flat) ratings 1 to 5 |
| `recommended` | BOOLEAN | Reviewer recommends the airline |
| `review` | VARCHAR | Free-text review |
| `updated_at` | TIMESTAMP | Load time |

## Step 3: Nulls & coverage

Null count and null % for every column, highest first. A high null % is real missing data, not something to fill.

In [40]:
null_check(T)

┌───────────────────┬────────┬──────────┐
│        col        │ n_null │ pct_null │
│      varchar      │ int64  │  double  │
├───────────────────┼────────┼──────────┤
│ seat_bed_length   │   3546 │    94.16 │
│ seat_privacy      │   3545 │    94.13 │
│ sleep_comfort     │   3545 │    94.13 │
│ seat_bed_width    │   3544 │    94.11 │
│ sitting_comfort   │   3543 │    94.08 │
│ power_supply      │   2382 │    63.25 │
│ seat_storage      │   1318 │     35.0 │
│ date_flown        │   1306 │    34.68 │
│ type_of_traveller │   1301 │    34.55 │
│ viewing_tv_screen │    980 │    26.02 │
│ aircraft_type     │    247 │     6.56 │
│ aisle_space       │    224 │     5.95 │
│ seat_recline      │    224 │     5.95 │
│ seat_legroom      │    223 │     5.92 │
│ seat_width        │    223 │     5.92 │
│ seat_layout       │      4 │     0.11 │
│ seat_type         │      3 │     0.08 │
│ customer_name     │      2 │     0.05 │
│ nationality       │      2 │     0.05 │
│ airline_id        │      0 │    

## Step 4: Profile by column type

Overview first (one line per column), then the type-specific checks.

In [41]:
profile_overview(T)

┌───────────────────┬───────────┬────────────┬─────────────────────┬─────────────────────┐
│        col        │ col_type  │ n_distinct │       min_val       │       max_val       │
│      varchar      │  varchar  │   int64    │       varchar       │       varchar       │
├───────────────────┼───────────┼────────────┼─────────────────────┼─────────────────────┤
│ review_id         │ BIGINT    │       3766 │ 1                   │ 3766                │
│ airline_id        │ BIGINT    │        205 │ 4                   │ 594                 │
│ type_of_traveller │ VARCHAR   │          5 │ 7                   │ 14                  │
│ seat_type         │ VARCHAR   │          5 │ 11                  │ 20                  │
│ aircraft_type     │ VARCHAR   │         63 │ 6                   │ 19                  │
│ seat_layout       │ VARCHAR   │         78 │ 3                   │ 15                  │
│ verify            │ BOOLEAN   │          2 │ NULL                │ NULL                │

### 4a. varchar

Distinct count, min/max length, most frequent value. Then the top values of categorical columns.

In [42]:
varchar_profile(T)

┌───────────────────┬────────────┬─────────┬─────────┬──────────────────────────────────────────┐
│        col        │ n_distinct │ min_len │ max_len │              most_frequent               │
│      varchar      │   int64    │  int64  │  int64  │                 varchar                  │
├───────────────────┼────────────┼─────────┼─────────┼──────────────────────────────────────────┤
│ type_of_traveller │          5 │       7 │      14 │ Solo Leisure                             │
│ seat_type         │          5 │      11 │      20 │ Economy Class                            │
│ aircraft_type     │         63 │       6 │      19 │ Airbus A330                              │
│ seat_layout       │         78 │       3 │      15 │ 3x3                                      │
│ customer_name     │       3029 │       8 │      23 │ Ronald Mitchell 8                        │
│ nationality       │         97 │       3 │      22 │ United Kingdom                           │
│ review            

In [43]:
for col in ["type_of_traveller", "seat_type"]:
    top_values(T, col)

┌────────────────┬───────┬────────┐
│     value      │   n   │  pct   │
│    varchar     │ int64 │ double │
├────────────────┼───────┼────────┤
│ NULL           │  1301 │  34.55 │
│ Solo Leisure   │   784 │  20.82 │
│ Couple Leisure │   660 │  17.53 │
│ Business       │   466 │  12.37 │
│ Family Leisure │   423 │  11.23 │
│ Leisure        │   132 │   3.51 │
└────────────────┴───────┴────────┘

┌──────────────────────┬───────┬────────┐
│        value         │   n   │  pct   │
│       varchar        │ int64 │ double │
├──────────────────────┼───────┼────────┤
│ Economy Class        │  2965 │  78.73 │
│ Premium Economy      │   567 │  15.06 │
│ Business Class       │   212 │   5.63 │
│ First Class          │    11 │   0.29 │
│ World Traveller Plus │     8 │   0.21 │
│ NULL                 │     3 │   0.08 │
└──────────────────────┴───────┴────────┘



In [44]:
top_values(T, "seat_layout", 80)

┌─────────────────┬───────┬────────┐
│      value      │   n   │  pct   │
│     varchar     │ int64 │ double │
├─────────────────┼───────┼────────┤
│ 3x3             │   895 │  23.77 │
│ 2x4x2           │   835 │  22.17 │
│ 3x4x3           │   750 │  19.92 │
│ 3x3x3           │   560 │  14.87 │
│ 2x3x2           │   216 │   5.74 │
│ 2x2             │   124 │   3.29 │
│ 1x2x1           │    92 │   2.44 │
│ 2x2x2           │    73 │   1.94 │
│ 2x5x2           │    20 │   0.53 │
│ 2x3             │    16 │   0.42 │
│ 3X3             │    16 │   0.42 │
│ 2-4-2           │    12 │   0.32 │
│ 3-3-3           │    11 │   0.29 │
│ 1x2             │    10 │   0.27 │
│ 3-4-3           │     9 │   0.24 │
│ 3x2             │     9 │   0.24 │
│ 3×3             │     8 │   0.21 │
│ 3×3×3           │     6 │   0.16 │
│ 3-3             │     5 │   0.13 │
│ 1-2-1           │     4 │   0.11 │
│ 2-3-2           │     4 │   0.11 │
│ 2X2             │     4 │   0.11 │
│ NULL            │     4 │   0.11 │
│

### 4b. numeric

Min, max, avg, median. For rating columns the histogram should show whole numbers 1 to 5 only.

In [45]:
numeric_stats(T)

┌───────────────────┬────────────┬─────────┬─────────┬─────────┬────────────┐
│        col        │ n_distinct │ min_val │ max_val │ avg_val │ median_val │
│      varchar      │   int64    │ double  │ double  │ double  │   double   │
├───────────────────┼────────────┼─────────┼─────────┼─────────┼────────────┤
│ review_id         │       3766 │     1.0 │  3766.0 │  1883.5 │     1883.5 │
│ airline_id        │        205 │     4.0 │   594.0 │  298.74 │      300.0 │
│ seat_legroom      │          5 │     1.0 │     5.0 │    2.84 │        3.0 │
│ seat_recline      │          5 │     1.0 │     5.0 │    2.66 │        3.0 │
│ seat_width        │          5 │     1.0 │     5.0 │    2.74 │        3.0 │
│ aisle_space       │          5 │     1.0 │     5.0 │    2.81 │        3.0 │
│ seat_storage      │          5 │     1.0 │     5.0 │    2.78 │        3.0 │
│ power_supply      │          5 │     1.0 │     5.0 │    3.52 │        4.0 │
│ viewing_tv_screen │          5 │     1.0 │     5.0 │    3.09 │

In [46]:
numeric_hist(T)

┌───────────────────┬────────┬────────┬────────┬────────┬────────┐
│        col        │  1.0   │  2.0   │  3.0   │  4.0   │  5.0   │
│      varchar      │ int128 │ int128 │ int128 │ int128 │ int128 │
├───────────────────┼────────┼────────┼────────┼────────┼────────┤
│ aisle_space       │    859 │    657 │    804 │    745 │    477 │
│ power_supply      │    240 │    117 │    215 │    310 │    502 │
│ seat_bed_length   │     59 │     22 │     37 │     22 │     80 │
│ seat_bed_width    │     76 │     21 │     31 │     31 │     63 │
│ seat_legroom      │   1063 │    500 │    587 │    716 │    677 │
│ seat_privacy      │     75 │     30 │     30 │     43 │     43 │
│ seat_recline      │   1033 │    624 │    793 │    698 │    394 │
│ seat_storage      │    695 │    392 │    511 │    456 │    394 │
│ seat_width        │    968 │    616 │    774 │    735 │    450 │
│ sitting_comfort   │     67 │     27 │     30 │     29 │     70 │
│ sleep_comfort     │     91 │     28 │     20 │     21 │     

### 4c. date

Min, max, rows before 1990 (placeholder dates such as 1970-01-01) and future dates.

In [47]:
date_stats(T)

┌────────────────┬─────────────────────┬─────────────────────┬───────────────┬──────────┬────────────┐
│      col       │       min_val       │       max_val       │ n_before_1990 │ n_future │ n_distinct │
│    varchar     │       varchar       │       varchar       │     int64     │  int64   │   int64    │
├────────────────┼─────────────────────┼─────────────────────┼───────────────┼──────────┼────────────┤
│ date_submitted │ 1970-01-01          │ 2026-06-11          │             6 │        0 │       1442 │
│ date_flown     │ 2014-08-01          │ 2026-06-01          │             0 │        0 │        137 │
│ updated_at     │ 2026-06-20 12:45:41 │ 2026-06-20 12:45:41 │             0 │        0 │          1 │
└────────────────┴─────────────────────┴─────────────────────┴───────────────┴──────────┴────────────┘


In [48]:
con.sql(f"select year(date_submitted) y, count(*) n from {T} group by y order by y")

┌───────┬───────┐
│   y   │   n   │
│ int64 │ int64 │
├───────┼───────┤
│  1970 │     6 │
│  2007 │     5 │
│  2008 │    81 │
│  2009 │   124 │
│  2010 │   126 │
│  2011 │   182 │
│  2012 │    24 │
│  2013 │   178 │
│  2014 │   391 │
│  2015 │   650 │
│  2016 │   507 │
│  2017 │   379 │
│  2018 │   254 │
│  2019 │   339 │
│  2020 │    91 │
│  2021 │    44 │
│  2022 │    88 │
│  2023 │   138 │
│  2024 │    83 │
│  2025 │    57 │
│  2026 │    19 │
└───────┴───────┘
     21 rows   

### 4d. boolean

Distribution of True / False / null.

In [49]:
bool_dist(T)

┌─────────────┬────────┬─────────┬────────┐
│     col     │ n_true │ n_false │ n_null │
│   varchar   │ int64  │  int64  │ int64  │
├─────────────┼────────┼─────────┼────────┤
│ verify      │    922 │    2844 │      0 │
│ recommended │   1641 │    2125 │      0 │
└─────────────┴────────┴─────────┴────────┘



## Step 5: Caveat hunting

Keys and duplicates first, then whitespace, then table-specific checks. Anything found here goes in the summary with a planned fix for dbt staging.

In [50]:
key_checks(T, 'review_id', {'airline_id': 'raw_airlines'})

┌────────────────┬─────────────────────┬───────────────────┐
│ duplicate_keys │ duplicate_full_rows │ orphan_airline_id │
│     int64      │        int64        │       int64       │
├────────────────┼─────────────────────┼───────────────────┤
│              0 │                   0 │                 0 │
└────────────────┴─────────────────────┴───────────────────┘



In [51]:
whitespace_check(T)

┌───────────────────┬──────────────┬────────────┐
│        col        │ n_whitespace │ n_one_char │
│      varchar      │    int64     │   int64    │
├───────────────────┼──────────────┼────────────┤
│ type_of_traveller │            0 │          0 │
│ seat_type         │            0 │          0 │
│ aircraft_type     │            0 │          0 │
│ seat_layout       │            0 │          0 │
│ customer_name     │            0 │          0 │
│ nationality       │            0 │          0 │
│ review            │            0 │          0 │
└───────────────────┴──────────────┴────────────┘



In [52]:
# date_flown after date_submitted would be impossible
con.sql(f"select count(*) as flown_after_submitted from {T} where date_flown > date_submitted")

┌───────────────────────┐
│ flown_after_submitted │
│         int64         │
├───────────────────────┤
│                     0 │
└───────────────────────┘

In [53]:
# epoch placeholder dates
con.sql(f"select review_id, airline_id, date_submitted from {T} where date_submitted = '1970-01-01'")

┌───────────┬────────────┬────────────────┐
│ review_id │ airline_id │ date_submitted │
│   int64   │   int64    │      date      │
├───────────┼────────────┼────────────────┤
│       567 │        115 │ 1970-01-01     │
│       568 │        115 │ 1970-01-01     │
│       569 │        115 │ 1970-01-01     │
│       570 │        115 │ 1970-01-01     │
│       571 │        115 │ 1970-01-01     │
│       572 │        115 │ 1970-01-01     │
└───────────┴────────────┴────────────────┘

In [54]:
con.sql("""
    SELECT seat_layout, COUNT(*) AS n
    FROM raw_seat_reviews
    WHERE seat_layout <> LOWER(seat_layout)         
       OR REGEXP_MATCHES(seat_layout, '[ -]')        
    GROUP BY 1
    ORDER BY n DESC
""").show(max_rows=100)

┌─────────────────┬───────┐
│   seat_layout   │   n   │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ 3X3             │    16 │
│ 2-4-2           │    12 │
│ 3-3-3           │    11 │
│ 3-4-3           │     9 │
│ 3-3             │     5 │
│ 1-2-1           │     4 │
│ 2X2             │     4 │
│ 2-3-2           │     4 │
│ 3X4X3           │     3 │
│ 3X3X3           │     3 │
│ 3x3x3 / 2x4x2   │     3 │
│ 3x4x3 / 2x4x2   │     3 │
│ 2 x 4 x 2       │     2 │
│ 3 x 3           │     2 │
│ 2x4x2 / 3x3x3   │     2 │
│ 2x3X2           │     2 │
│ 2 x 2           │     2 │
│ 2-2-2           │     2 │
│ 2X4X2           │     2 │
│ 2 by 2          │     1 │
│ 3x3 / 2x2       │     1 │
│ 3x3, 2x2        │     1 │
│ 2x4x2, 3x3x3    │     1 │
│ 2x2, 3x3        │     1 │
│ 1X2X1           │     1 │
│ 3x4x3 & 2x4x2   │     1 │
│ 3x4x3 & 3x3x3   │     1 │
│ 3x4x3 / 3x3x3   │     1 │
│ Premium Economy │     1 │
│ 2x2x2 and 3x3x3 │     1 │
│ 2x3x2 and 2x4x2 │     1 │
│ 3x4x3 / 2x3x2   │ 

In [55]:
con.sql("""
    WITH normalized AS (
        SELECT
            seat_layout,
            COALESCE(
                NULLIF(
                    ARRAY_TO_STRING(REGEXP_EXTRACT_ALL(seat_layout, '\\d+'), 'x'),
                    ''
                ),
                LOWER(TRIM(seat_layout))   
            ) AS normalized_value
        FROM raw_seat_reviews
    )
    SELECT
        normalized_value,
        LIST(DISTINCT seat_layout)   AS original_variants,
        COUNT(DISTINCT seat_layout)  AS n_variants
    FROM normalized
    GROUP BY 1
    HAVING COUNT(DISTINCT seat_layout) > 1
    ORDER BY n_variants DESC
""").show(max_rows=100, max_width=200)

┌──────────────────┬────────────────────────────────────────────────┬────────────┐
│ normalized_value │               original_variants                │ n_variants │
│     varchar      │                   varchar[]                    │   int64    │
├──────────────────┼────────────────────────────────────────────────┼────────────┤
│ 3x3              │ [3 x 3, 3X3, 3×3, 3x3, 3*3, 3 + 3, 3 X 3, 3-3] │          8 │
│ 2x4x2            │ [2×4×2, 2X4X2, 2X4x2, 2 x 4 x 2, 2x4x2, 2-4-2] │          6 │
│ 3x3x3            │ [3-3-3, 3x3x3, 3*3*3, 3×3×3, 3 - 3 - 3, 3X3X3] │          6 │
│ 2x2              │ [2 x 2, 2X2, 2 by 2, 2xax2, 2x2]               │          5 │
│ 2x2x2            │ [2+2+2, 2-2-2, 2x2x2, 2X2X2]                   │          4 │
│ 2x3x2            │ [2-3-2, 2x3x2, 2x3X2, 2×3×2]                   │          4 │
│ 3x4x3            │ [3X4X3, 3-4-3, 3x4x3, 3×4×3]                   │          4 │
│ 3x3x3x2x4x2      │ [3x3x3  / 2x4x2, 3x3x3 / 2x4x2, 3-3-3 ; 2-4-2] │          3 │
│ 1x

In [56]:
con.sql("""
    SELECT nationality, COUNT(*) AS n
    FROM raw_seat_reviews
    WHERE REGEXP_MATCHES(nationality, '[0-9]')
    GROUP BY nationality
    ORDER BY n DESC
""").show(max_rows=100)

┌─────────────┬───────┐
│ nationality │   n   │
│   varchar   │ int64 │
├─────────────┼───────┤
│ 71J         │     2 │
│ 3x3x3       │     1 │
└─────────────┴───────┘



In [57]:
con.sql("""
    SELECT customer_name
    FROM raw_seat_reviews
    USING SAMPLE 8
""").show()

┌─────────────────────┐
│    customer_name    │
│       varchar       │
├─────────────────────┤
│ George Thomas 11    │
│ Aisha Hernandez 22  │
│ Mei Baker 26        │
│ Jason Kim           │
│ Patricia Schmidt 17 │
│ Hassan Haddad 21    │
│ Steven Hernandez 13 │
│ Diego Rivera 13     │
└─────────────────────┘



In [58]:
con.sql("""
    SELECT COUNT(*) AS names_with_trailing_number
    FROM raw_seat_reviews
    WHERE REGEXP_MATCHES(customer_name, ' [0-9]+$')
""").show()

┌────────────────────────────┐
│ names_with_trailing_number │
│           int64            │
├────────────────────────────┤
│                       3635 │
└────────────────────────────┘



In [59]:
dup_text(T, "review")

┌────────────────────────────┐
│ rows_with_duplicate_review │
│           int64            │
├────────────────────────────┤
│                        672 │
└────────────────────────────┘



## Step 6: Summary of `raw_seat_reviews`

**Purpose:** Fact table of seat reviews, one row per review. Joins to `raw_airlines` on `airline_id`.

**Shape:** 3,766 rows x 26 columns.

| Column group | Columns | DuckDB type | Meaning | Null % | Profile |
|---|---|---|---|---|---|
| Keys | `review_id`, `airline_id` | BIGINT | Review id and airline | 0% | `review_id` 1-3766, all unique; 205 distinct airlines |
| Traveller | `type_of_traveller` | VARCHAR | Trip type | 34.5% | 5 values, length 7-14 |
| Seat | `seat_type`, `aircraft_type`, `seat_layout` | VARCHAR | Cabin class, aircraft, seat configuration | 0.1% / 6.6% / 0.1% | 5 / 63 / 78 distinct; max length 20 / 19 / 15 |
| Flags | `verify`, `recommended` | BOOLEAN | Verified review; recommends airline | 0% | True and False both present |
| Dates | `date_submitted`, `date_flown` | DATE | Review post date; flight month | 0% / 34.7% | submitted 1970-01-01 to 2026-06-11; flown 2014-08-01 to 2026-06-01 |
| Reviewer | `customer_name`, `nationality` | VARCHAR | Reviewer name and country | 0.1% | 3,029 / 97 distinct; max length 23 / 22 |
| Ratings (core) | `seat_legroom`, `seat_recline`, `seat_width`, `aisle_space` | DOUBLE | Rating 1-5 | ~5.9% | integers 1-5 only |
| Ratings (optional) | `seat_storage`, `viewing_tv_screen`, `power_supply` | DOUBLE | Rating 1-5 | 26-63% | integers 1-5 only |
| Ratings (premium) | `sleep_comfort`, `sitting_comfort`, `seat_bed_width`, `seat_bed_length`, `seat_privacy` | DOUBLE | Rating 1-5 for lie-flat seats | ~94% | integers 1-5 only |
| Text | `review` | VARCHAR | Free-text review | 0% | length 32-3,501 |
| Metadata | `updated_at` | TIMESTAMP | Load time | 0% | one value for all rows |

**Caveat checks:**
- Duplicate `review_id`: 0. Orphaned `airline_id`: 0. Full-row duplicates: 0.
- Leading/trailing whitespace in text columns: 0.
- `date_flown` is never later than `date_submitted`. No future dates.
- **`date_submitted = 1970-01-01`: 6 rows** (`review_id` 567-572, all airline 115). Epoch placeholder, not a real date.
- **`seat_layout` is badly inconsistent: 149 rows, 59 variants not matching `NxN(xN)`.** 78 raw values collapse to 36 after normalisation. Variants include uppercase `X`, `-`, `*`, `+`, `x`, spaces, `2 by 2`, the typo `2xax2`, and multi-values like `2x2x2 and 3x3x3`. Two rows hold text that is not a layout at all (`Aisle`, `Premium Economy`).
- **Wrong-column values in `nationality`:** `3x3x3` (1 row, a seat layout) and `71J` (2 rows).
- **`customer_name` ends with a number in 3,635 of 3,766 rows** (e.g. `Andrew Patel 14`). The suffix is not part of a real name.
- **Duplicate `review` text:** 672 rows share repeated texts (different `review_id`).
- `type_of_traveller` mixes `Leisure` with `Solo/Couple/Family Leisure`; `seat_type` has `World Traveller Plus` (8 rows), an airline brand name for Premium Economy.
- The five premium rating columns are ~94% null by design (only business/first class). `updated_at` carries no information.

**Conclusion:** Keys and the join to `raw_airlines` are sound. Clean up `date_submitted` (1970), `seat_layout` (format and stray text), `nationality` (3 bad values), `customer_name` (numeric suffix) and duplicate review text. High nulls in `type_of_traveller`, `date_flown` 
and the optional ratings are real missing data and should not be filled.

# Table 4: `raw_lounge_reviews`

**Role:** Fact table of lounge reviews, one row per review. Joins to `raw_airlines` on `airline_id`.

**Steps:** 1 Shape & schema, 2 Column descriptions, 3 Nulls & coverage, 4 Profile by type (4a varchar, 4b numeric, 4c date, 4d boolean), 5 Caveat hunting, 6 Summary.

In [60]:
T = "raw_lounge_reviews"

## Step 1: Shape & schema

How many rows and columns, what types DuckDB inferred, and what the data looks like.

In [61]:
con.sql(f"describe {T}").show(max_rows=40, max_width=200)

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ review_id         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ airline_id        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ lounge_name       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ airport           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ type_of_lounge    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ type_of_traveller │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ verify            │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_submitted    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ date_visit        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_n

In [62]:
con.sql(f"select count(*) as n_rows from {T}")

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   5087 │
└────────┘

In [63]:
con.sql(f"select * from {T} limit 10").show(max_width=200)

┌───────────┬────────────┬────────────────┬─────────────────────────┬────────────────┬───┬───────────┬───────────────────┬───────────────┬─────────────┬─────────────────────────┬─────────────────────┐
│ review_id │ airline_id │  lounge_name   │         airport         │ type_of_lounge │ … │ washrooms │ wifi_connectivity │ staff_service │ recommended │         review          │     updated_at      │
│   int64   │   int64    │    varchar     │         varchar         │    varchar     │ … │  double   │      double       │    double     │   boolean   │         varchar         │      timestamp      │
├───────────┼────────────┼────────────────┼─────────────────────────┼────────────────┼───┼───────────┼───────────────────┼───────────────┼─────────────┼─────────────────────────┼─────────────────────┤
│         1 │          9 │ JFK Terminal 7 │ New York JFK Airport    │ Business Class │ … │       4.0 │               5.0 │           4.0 │ true        │ This is a new lounge f… │ 2026-06-20 12:45:

## Step 2: Column descriptions

One line per column. Meanings are inferred from the column name and the data, so correct them if you know better.

| Column | Type | Meaning |
|---|---|---|
| `review_id` | BIGINT | Review id (1 to 5087) |
| `airline_id` | BIGINT | Airline reviewed, joins to `raw_airlines` |
| `lounge_name` | VARCHAR | Name of the lounge, free text |
| `airport` | VARCHAR | Airport where the lounge is |
| `type_of_lounge` | VARCHAR | Lounge access class (Business Class, First Class, Frequent Flyer, Members, Pay to Use) |
| `type_of_traveller` | VARCHAR | Trip type |
| `verify` | BOOLEAN | Verified review |
| `date_submitted` | DATE | Date the review was posted |
| `date_visit` | DATE | Month of the lounge visit (first day of month) |
| `customer_name` | VARCHAR | Reviewer name |
| `nationality` | VARCHAR | Reviewer country |
| `comfort`, `cleanliness`, `bar_and_beverages`, `catering`, `washrooms`, `wifi_connectivity`, `staff_service` | DOUBLE | Rating 1 to 5 per aspect |
| `recommended` | BOOLEAN | Reviewer recommends the lounge |
| `review` | VARCHAR | Free-text review |
| `updated_at` | TIMESTAMP | Load time |

## Step 3: Nulls & coverage

Null count and null % for every column, highest first. A high null % is real missing data, not something to fill.

In [64]:
null_check(T)

┌───────────────────┬────────┬──────────┐
│        col        │ n_null │ pct_null │
│      varchar      │ int64  │  double  │
├───────────────────┼────────┼──────────┤
│ lounge_name       │   2411 │     47.4 │
│ date_visit        │   2290 │    45.02 │
│ type_of_traveller │   1453 │    28.56 │
│ washrooms         │    626 │    12.31 │
│ wifi_connectivity │    443 │     8.71 │
│ type_of_lounge    │    304 │     5.98 │
│ bar_and_beverages │    152 │     2.99 │
│ staff_service     │    117 │      2.3 │
│ airport           │     99 │     1.95 │
│ catering          │     88 │     1.73 │
│ cleanliness       │     28 │     0.55 │
│ comfort           │     24 │     0.47 │
│ customer_name     │      1 │     0.02 │
│ nationality       │      1 │     0.02 │
│ airline_id        │      0 │      0.0 │
│ date_submitted    │      0 │      0.0 │
│ recommended       │      0 │      0.0 │
│ review            │      0 │      0.0 │
│ review_id         │      0 │      0.0 │
│ updated_at        │      0 │    

## Step 4: Profile by column type

Overview first (one line per column), then the type-specific checks.

In [65]:
profile_overview(T)

┌───────────────────┬───────────┬────────────┬─────────────────────┬─────────────────────┐
│        col        │ col_type  │ n_distinct │       min_val       │       max_val       │
│      varchar      │  varchar  │   int64    │       varchar       │       varchar       │
├───────────────────┼───────────┼────────────┼─────────────────────┼─────────────────────┤
│ review_id         │ BIGINT    │       5087 │ 1                   │ 5087                │
│ airline_id        │ BIGINT    │        159 │ 2                   │ 595                 │
│ lounge_name       │ VARCHAR   │       1089 │ 1                   │ 60                  │
│ airport           │ VARCHAR   │        320 │ 11                  │ 31                  │
│ type_of_lounge    │ VARCHAR   │          5 │ 7                   │ 14                  │
│ type_of_traveller │ VARCHAR   │          1 │ 8                   │ 8                   │
│ verify            │ BOOLEAN   │          2 │ NULL                │ NULL                │

### 4a. varchar

Distinct count, min/max length, most frequent value. Then the top values of categorical columns.

In [66]:
varchar_profile(T)

┌───────────────────┬────────────┬─────────┬─────────┬──────────────────────────────────────────┐
│        col        │ n_distinct │ min_len │ max_len │              most_frequent               │
│      varchar      │   int64    │  int64  │  int64  │                 varchar                  │
├───────────────────┼────────────┼─────────┼─────────┼──────────────────────────────────────────┤
│ lounge_name       │       1089 │       1 │      60 │ Admirals Club                            │
│ airport           │        320 │      11 │      31 │ London Heathrow Airport                  │
│ type_of_lounge    │          5 │       7 │      14 │ Business Class                           │
│ type_of_traveller │          1 │       8 │       8 │ Business                                 │
│ customer_name     │       3122 │       9 │      24 │ Aisha Costa 25                           │
│ nationality       │         98 │       4 │      20 │ United Kingdom                           │
│ review            

In [67]:
for col in ["type_of_traveller", "type_of_lounge"]:
    top_values(T, col)

┌──────────┬───────┬────────┐
│  value   │   n   │  pct   │
│ varchar  │ int64 │ double │
├──────────┼───────┼────────┤
│ Business │  3634 │  71.44 │
│ NULL     │  1453 │  28.56 │
└──────────┴───────┴────────┘

┌────────────────┬───────┬────────┐
│     value      │   n   │  pct   │
│    varchar     │ int64 │ double │
├────────────────┼───────┼────────┤
│ Business Class │  3634 │  71.44 │
│ First Class    │   501 │   9.85 │
│ Frequent Flyer │   412 │    8.1 │
│ NULL           │   304 │   5.98 │
│ Members        │   135 │   2.65 │
│ Pay to Use     │   101 │   1.99 │
└────────────────┴───────┴────────┘



In [68]:
top_values(T, "lounge_name", 15)

┌───────────────────┬───────┬────────┐
│       value       │   n   │  pct   │
│      varchar      │ int64 │ double │
├───────────────────┼───────┼────────┤
│ NULL              │  2411 │   47.4 │
│ Admirals Club     │    89 │   1.75 │
│ Maple Leaf        │    67 │   1.32 │
│ United Club       │    61 │    1.2 │
│ Senator Lounge    │    54 │   1.06 │
│ Galleries Lounge  │    53 │   1.04 │
│ Terraces Lounge   │    39 │   0.77 │
│ Business Lounge   │    30 │   0.59 │
│ Golden Lounge     │    29 │   0.57 │
│ Royal Silk        │    29 │   0.57 │
│ Qantas Club       │    27 │   0.53 │
│ SkyClub           │    27 │   0.53 │
│ The Wing          │    26 │   0.51 │
│ Crown Lounge      │    25 │   0.49 │
│ SilverKris Lounge │    25 │   0.49 │
└───────────────────┴───────┴────────┘
  15 rows                  3 columns



In [69]:
top_values(T, "airport", 10)

┌──────────────────────────────┬───────┬────────┐
│            value             │   n   │  pct   │
│           varchar            │ int64 │ double │
├──────────────────────────────┼───────┼────────┤
│ London Heathrow Airport      │   410 │   8.06 │
│ Istanbul Airport             │   234 │    4.6 │
│ Bangkok Suvarnabhumi Airport │   193 │   3.79 │
│ Hong Kong Airport            │   161 │   3.16 │
│ Singapore Changi Airport     │   148 │   2.91 │
│ Dubai Airport                │   132 │   2.59 │
│ Sydney Airport               │   126 │   2.48 │
│ NULL                         │    99 │   1.95 │
│ KLIA Kuala Lumpur Airport    │    98 │   1.93 │
│ New York JFK Airport         │    98 │   1.93 │
└──────────────────────────────┴───────┴────────┘
  10 rows                             3 columns



In [70]:
top_values(T, "nationality", 10)

┌────────────────┬───────┬────────┐
│     value      │   n   │  pct   │
│    varchar     │ int64 │ double │
├────────────────┼───────┼────────┤
│ United Kingdom │  1566 │  30.78 │
│ United States  │   706 │  13.88 │
│ Australia      │   700 │  13.76 │
│ Canada         │   212 │   4.17 │
│ Germany        │   193 │   3.79 │
│ Thailand       │   132 │   2.59 │
│ Hong Kong      │    86 │   1.69 │
│ Switzerland    │    85 │   1.67 │
│ New Zealand    │    73 │   1.44 │
│ Singapore      │    71 │    1.4 │
└────────────────┴───────┴────────┘
  10 rows               3 columns



### 4b. numeric

Min, max, avg, median. For rating columns the histogram should show whole numbers 1 to 5 only.

In [71]:
numeric_stats(T)

┌───────────────────┬────────────┬─────────┬─────────┬─────────┬────────────┐
│        col        │ n_distinct │ min_val │ max_val │ avg_val │ median_val │
│      varchar      │   int64    │ double  │ double  │ double  │   double   │
├───────────────────┼────────────┼─────────┼─────────┼─────────┼────────────┤
│ review_id         │       5087 │     1.0 │  5087.0 │  2544.0 │     2544.0 │
│ airline_id        │        159 │     2.0 │   595.0 │  297.44 │      305.0 │
│ comfort           │          5 │     1.0 │     5.0 │     3.4 │        4.0 │
│ cleanliness       │          5 │     1.0 │     5.0 │    3.72 │        4.0 │
│ bar_and_beverages │          5 │     1.0 │     5.0 │    3.29 │        4.0 │
│ catering          │          5 │     1.0 │     5.0 │    2.92 │        3.0 │
│ washrooms         │          5 │     1.0 │     5.0 │    3.28 │        3.0 │
│ wifi_connectivity │          5 │     1.0 │     5.0 │    3.65 │        4.0 │
│ staff_service     │          5 │     1.0 │     5.0 │    3.38 │

In [72]:
numeric_hist(T)

┌───────────────────┬────────┬────────┬────────┬────────┬────────┐
│        col        │  1.0   │  2.0   │  3.0   │  4.0   │  5.0   │
│      varchar      │ int128 │ int128 │ int128 │ int128 │ int128 │
├───────────────────┼────────┼────────┼────────┼────────┼────────┤
│ bar_and_beverages │    772 │    700 │    994 │   1244 │   1225 │
│ catering          │   1308 │    744 │    913 │   1086 │    948 │
│ cleanliness       │    455 │    508 │    877 │   1394 │   1825 │
│ comfort           │    635 │    712 │   1003 │   1411 │   1302 │
│ staff_service     │    806 │    629 │    898 │   1141 │   1496 │
│ washrooms         │    730 │    571 │    941 │   1167 │   1052 │
│ wifi_connectivity │    506 │    392 │    826 │   1418 │   1502 │
└───────────────────┴────────┴────────┴────────┴────────┴────────┘



### 4c. date

Min, max, rows before 1990 (placeholder dates such as 1970-01-01) and future dates.

In [73]:
date_stats(T)

┌────────────────┬─────────────────────┬─────────────────────┬───────────────┬──────────┬────────────┐
│      col       │       min_val       │       max_val       │ n_before_1990 │ n_future │ n_distinct │
│    varchar     │       varchar       │       varchar       │     int64     │  int64   │   int64    │
├────────────────┼─────────────────────┼─────────────────────┼───────────────┼──────────┼────────────┤
│ date_submitted │ 2006-05-09          │ 2026-06-14          │             0 │        0 │       1736 │
│ date_visit     │ 2014-01-01          │ 2026-06-01          │             0 │        0 │        137 │
│ updated_at     │ 2026-06-20 12:45:41 │ 2026-06-20 12:45:41 │             0 │        0 │          1 │
└────────────────┴─────────────────────┴─────────────────────┴───────────────┴──────────┴────────────┘



In [74]:
con.sql(f"select year(date_submitted) y, count(*) n from {T} group by 1 order by 1")

┌───────┬───────┐
│   y   │   n   │
│ int64 │ int64 │
├───────┼───────┤
│  2006 │     8 │
│  2007 │    43 │
│  2008 │   103 │
│  2009 │   173 │
│  2010 │   180 │
│  2011 │   267 │
│  2012 │   342 │
│  2013 │   449 │
│  2014 │   424 │
│  2015 │   708 │
│  2016 │   621 │
│  2017 │   533 │
│  2018 │   333 │
│  2019 │   329 │
│  2020 │    65 │
│  2021 │    39 │
│  2022 │   138 │
│  2023 │   153 │
│  2024 │    87 │
│  2025 │    70 │
│  2026 │    22 │
└───────┴───────┘
     21 rows   

### 4d. boolean

Distribution of True / False / null.

In [75]:
bool_dist(T)

┌─────────────┬────────┬─────────┬────────┐
│     col     │ n_true │ n_false │ n_null │
│   varchar   │ int64  │  int64  │ int64  │
├─────────────┼────────┼─────────┼────────┤
│ verify      │   1101 │    3986 │      0 │
│ recommended │   2653 │    2434 │      0 │
└─────────────┴────────┴─────────┴────────┘



In [76]:
con.sql(f"select verify, recommended, count(*) n from {T} group by all order by n desc")

┌─────────┬─────────────┬───────┐
│ verify  │ recommended │   n   │
│ boolean │   boolean   │ int64 │
├─────────┼─────────────┼───────┤
│ false   │ false       │  2039 │
│ false   │ true        │  1947 │
│ true    │ true        │   706 │
│ true    │ false       │   395 │
└─────────┴─────────────┴───────┘

## Step 5: Caveat hunting

Keys and duplicates first, then whitespace, then table-specific checks. Anything found here goes in the summary with a planned fix for dbt staging.

In [77]:
key_checks(T, 'review_id', {'airline_id': 'raw_airlines'})

┌────────────────┬─────────────────────┬───────────────────┐
│ duplicate_keys │ duplicate_full_rows │ orphan_airline_id │
│     int64      │        int64        │       int64       │
├────────────────┼─────────────────────┼───────────────────┤
│              0 │                   0 │                 0 │
└────────────────┴─────────────────────┴───────────────────┘



In [78]:
whitespace_check(T)

┌───────────────────┬──────────────┬────────────┐
│        col        │ n_whitespace │ n_one_char │
│      varchar      │    int64     │   int64    │
├───────────────────┼──────────────┼────────────┤
│ lounge_name       │            0 │          2 │
│ airport           │            0 │          0 │
│ type_of_lounge    │            0 │          0 │
│ type_of_traveller │            0 │          0 │
│ customer_name     │            0 │          0 │
│ nationality       │            0 │          0 │
│ review            │            0 │          0 │
└───────────────────┴──────────────┴────────────┘



In [79]:
# date_visit after date_submitted would be impossible
con.sql(f"select count(*) as visit_after_submitted from {T} where date_visit > date_submitted")

┌───────────────────────┐
│ visit_after_submitted │
│         int64         │
├───────────────────────┤
│                     0 │
└───────────────────────┘

In [80]:
# reviews submitted before 2010 (site started in 2006, so plausible; just look)
con.sql(f"select review_id, airline_id, date_submitted from {T} where date_submitted < '2010-01-01' order by 3 limit 20")

┌───────────┬────────────┬────────────────┐
│ review_id │ airline_id │ date_submitted │
│   int64   │   int64    │      date      │
├───────────┼────────────┼────────────────┤
│       129 │         38 │ 2006-05-09     │
│      2564 │        251 │ 2006-09-28     │
│      2342 │        265 │ 2006-10-14     │
│      4416 │        539 │ 2006-12-10     │
│      5087 │        419 │ 2006-12-10     │
│      2392 │        281 │ 2006-12-14     │
│      2393 │        281 │ 2006-12-14     │
│      2563 │        251 │ 2006-12-29     │
│      2512 │        173 │ 2007-01-05     │
│       728 │        115 │ 2007-01-06     │
│      3902 │        416 │ 2007-01-13     │
│       128 │         38 │ 2007-01-25     │
│      3497 │        433 │ 2007-02-13     │
│      3901 │        416 │ 2007-03-11     │
│      2391 │        281 │ 2007-03-24     │
│      4019 │        482 │ 2007-03-27     │
│      4199 │        542 │ 2007-04-15     │
│      2606 │        307 │ 2007-04-17     │
│      2605 │        307 │ 2007-

In [81]:
# lounge_name written with different case: same lounge, several spellings
con.sql(f"""
    select lower(trim(lounge_name)) k, list(distinct lounge_name) variants
    from {T} group by 1 having count(distinct lounge_name) > 1 limit 20
""").show(max_width=200)

┌────────────────────────┬──────────────────────────────────────────────────┐
│           k            │                     variants                     │
│        varchar         │                    varchar[]                     │
├────────────────────────┼──────────────────────────────────────────────────┤
│ dnata t1               │ [Dnata T1, DNATA T1]                             │
│ ebl skylounge          │ [EBL Skylounge, EBL SkyLounge]                   │
│ crowne lounge          │ [Crowne lounge, Crowne Lounge]                   │
│ senator lounge         │ [Senator Lounge, Senator lounge]                 │
│ crown lounge           │ [Crown Lounge, Crown lounge]                     │
│ skyteam lounge         │ [Skyteam Lounge, Skyteam lounge, SkyTeam Lounge] │
│ platinum lounge        │ [Platinum Lounge, Platinum lounge]               │
│ casa italia            │ [Casa Italia, casa italia]                       │
│ pride lounge           │ [Pride lounge, Pride Lounge]         

In [82]:
# lounge_name format: ALL-CAPS, '... LOUNGE REVIEW' suffix (scraping leftover), distinct count raw vs normalised
con.sql(f"""
    select count(*) filter (where lounge_name = upper(lounge_name))    as n_all_caps,
           count(*) filter (where lounge_name ilike '% lounge review') as n_review_suffix,
           count(distinct lounge_name)                                 as n_distinct_raw,
           count(distinct lower(trim(lounge_name)))                    as n_distinct_normalised
    from {T}
""").show(max_width=200)

┌────────────┬─────────────────┬────────────────┬───────────────────────┐
│ n_all_caps │ n_review_suffix │ n_distinct_raw │ n_distinct_normalised │
│   int64    │      int64      │     int64      │         int64         │
├────────────┼─────────────────┼────────────────┼───────────────────────┤
│         52 │              19 │           1089 │                  1015 │
└────────────┴─────────────────┴────────────────┴───────────────────────┘



In [83]:
con.sql(f"select lounge_name, length(lounge_name) l from {T} order by l desc nulls last limit 5").show(max_width=200)

┌──────────────────────────────────────────────────────────────┬───────┐
│                         lounge_name                          │   l   │
│                           varchar                            │ int64 │
├──────────────────────────────────────────────────────────────┼───────┤
│ DELTA AIR LINES SKYCLUB FIRST & BUSINESS CLASS LOUNGE REVIEW │    60 │
│ QATAR AIRWAYS PREMIUM TERMINAL BUSINESS CLASS LOUNGE REVIEW  │    59 │
│ QATAR AIRWAYS PREMIUM TERMINAL BUSINESS CLASS LOUNGE REVIEW  │    59 │
│ AIR NEW ZEALAND BUSINESS CLASS (INT'L) LOUNGE REVIEW         │    52 │
│ IBERIA "SALA VIP" BUSINESS CLASS LOUNGE REVIEW               │    46 │
└──────────────────────────────────────────────────────────────┴───────┘



In [84]:
con.sql(f"select review_id, airline_id, lounge_name, airport, type_of_lounge from {T} where length(lounge_name) = 1")

┌───────────┬────────────┬─────────────┬─────────────────────────┬────────────────┐
│ review_id │ airline_id │ lounge_name │         airport         │ type_of_lounge │
│   int64   │   int64    │   varchar   │         varchar         │    varchar     │
├───────────┼────────────┼─────────────┼─────────────────────────┼────────────────┤
│       899 │         17 │ 2           │ Mexico City Airport     │ Frequent Flyer │
│      2020 │        254 │ =           │ Beijing Capital Airport │ Business Class │
└───────────┴────────────┴─────────────┴─────────────────────────┴────────────────┘

In [85]:
# lounge_name and airport missing together or separately
con.sql(f"""
    select lounge_name is null as lounge_null, airport is null as airport_null, count(*) as n
    from {T} group by all order by n desc
""")

┌─────────────┬──────────────┬───────┐
│ lounge_null │ airport_null │   n   │
│   boolean   │   boolean    │ int64 │
├─────────────┼──────────────┼───────┤
│ false       │ false        │  2626 │
│ true        │ false        │  2362 │
│ false       │ true         │    50 │
│ true        │ true         │    49 │
└─────────────┴──────────────┴───────┘

In [86]:
# does type_of_traveller carry information beyond type_of_lounge?
con.sql(f"select type_of_traveller, type_of_lounge, count(*) n from {T} group by all order by n desc")

┌───────────────────┬────────────────┬───────┐
│ type_of_traveller │ type_of_lounge │   n   │
│      varchar      │    varchar     │ int64 │
├───────────────────┼────────────────┼───────┤
│ Business          │ Business Class │  3634 │
│ NULL              │ First Class    │   501 │
│ NULL              │ Frequent Flyer │   412 │
│ NULL              │ NULL           │   304 │
│ NULL              │ Members        │   135 │
│ NULL              │ Pay to Use     │   101 │
└───────────────────┴────────────────┴───────┘

In [87]:
# nationality holding digits, and customer_name ending with a number (same pattern as raw_seat_reviews)
con.sql(f"""
    select count(*) filter (where regexp_matches(nationality, '[0-9]'))      as nationality_with_digit,
           count(*) filter (where regexp_matches(customer_name, ' [0-9]+$')) as names_with_trailing_number
    from {T}
""")

┌────────────────────────┬────────────────────────────┐
│ nationality_with_digit │ names_with_trailing_number │
│         int64          │           int64            │
├────────────────────────┼────────────────────────────┤
│                      0 │                       4893 │
└────────────────────────┴────────────────────────────┘

In [88]:
dup_text(T, "review")

┌────────────────────────────┐
│ rows_with_duplicate_review │
│           int64            │
├────────────────────────────┤
│                        481 │
└────────────────────────────┘



## Step 6: Summary of `raw_lounge_reviews`

**Purpose:** Fact table of lounge reviews, one row per review. Joins to `raw_airlines` on `airline_id`.

**Shape:** 5,087 rows x 21 columns.

| Column group | Columns | DuckDB type | Meaning | Null % | Profile |
|---|---|---|---|---|---|
| Keys | `review_id`, `airline_id` | BIGINT | Review id and airline | 0% | `review_id` 1-5087, all unique; 159 distinct airlines |
| Lounge | `lounge_name`, `airport`, `type_of_lounge` | VARCHAR | Lounge, airport, access class | 47.4% / 1.95% / 5.98% | 1,089 / 320 / 5 distinct; max length 60 / 31 / 14 |
| Traveller | `type_of_traveller` | VARCHAR | Trip type | 28.6% | **1 value only** (`Business`) |
| Flags | `verify`, `recommended` | BOOLEAN | Verified review; recommends lounge | 0% | True and False both present |
| Dates | `date_submitted`, `date_visit` | DATE | Review post date; visit month | 0% / 45.0% | submitted 2006-05-09 to 2026-06-14; visit 2014-01-01 to 2026-06-01 |
| Reviewer | `customer_name`, `nationality` | VARCHAR | Reviewer name and country | 0.02% | 3,122 / 98 distinct |
| Ratings | `comfort`, `cleanliness`, `bar_and_beverages`, `catering`, `washrooms`, `wifi_connectivity`, `staff_service` | DOUBLE | Rating 1-5 | 0.5% to 12.3% | integers 1-5 only |
| Text | `review` | VARCHAR | Free-text review | 0% | length 91-3,479 |
| Metadata | `updated_at` | TIMESTAMP | Load time | 0% | one value for all rows |

**Caveat checks:**
- Duplicate `review_id`: 0. Orphaned `airline_id`: 0. Full-row duplicates: 0.
- Leading/trailing whitespace: 0 in every text column.
- No future dates, no epoch placeholder (earliest `date_submitted` is 2006-05-09), `date_visit` never later than `date_submitted`.
- Rating columns hold integers 1-5 only.
- **`lounge_name` is null in 47.4% of rows (2,411).** 99 rows also have no `airport`.
- **`lounge_name` format:** 1,089 raw values collapse to 1,015 after lower/trim; 52 rows are ALL CAPS; 19 rows end with the scraping leftover `LOUNGE REVIEW`; 2 rows are a single character (`=`).
- **`type_of_traveller` has a single value, `Business`, and it only appears together with `type_of_lounge = Business Class` (3,634 rows).** It carries no extra information.
- **`customer_name` ends with a number in 4,893 of 5,087 rows** (same pattern as seat reviews). The suffix is not part of a real name.
- **Duplicate `review` text:** 481 rows share repeated texts (different `review_id`).
- `nationality` has no digit values (the bad values found in seat reviews do not appear here).
- `updated_at` carries no information.

**Conclusion:** Keys and the join to `raw_airlines` are sound. In staging: normalise `lounge_name` (trim, case, drop `LOUNGE REVIEW` suffix, drop `=`), strip the numeric suffix from `customer_name`, decide how to treat duplicate review text, and drop `type_of_traveller` and `updated_at` or keep them as-is with a note. High nulls in `lounge_name`, `date_visit`, `type_of_traveller` are real missing data and should not be filled.

# Table 5: `raw_airport_reviews`

**Role:** Fact table of airport reviews, one row per review. Joins to `raw_airports` on `airport_id`.

**Steps:** 1 Shape & schema, 2 Column descriptions, 3 Nulls & coverage, 4 Profile by type (4a varchar, 4b numeric, 4c date, 4d boolean), 5 Caveat hunting, 6 Summary.

In [89]:
T = "raw_airport_reviews"

## Step 1: Shape & schema

How many rows and columns, what types DuckDB inferred, and what the data looks like.

In [90]:
con.sql(f"describe {T}").show(max_rows=40, max_width=200)

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ review_id             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ airport_id            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ verify                │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_submitted        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ date_visit            │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ nationality           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ experience_at_airport │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ type_of_traveller     │ VARCHAR     │ 

In [91]:
con.sql(f"select count(*) as n_rows from {T}")

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│  49505 │
└────────┘

In [92]:
con.sql(f"select * from {T} limit 10").show(max_width=200)

┌───────────┬────────────┬─────────┬────────────────┬────────────┬────────────────────┬───┬───────────────┬───────────────────┬─────────────┬────────────────────────────────────┬─────────────────────┐
│ review_id │ airport_id │ verify  │ date_submitted │ date_visit │   customer_name    │ … │ airport_staff │ wifi_connectivity │ recommended │               review               │     updated_at      │
│   int64   │   int64    │ boolean │      date      │    date    │      varchar       │ … │    double     │      double       │   boolean   │              varchar               │      timestamp      │
├───────────┼────────────┼─────────┼────────────────┼────────────┼────────────────────┼───┼───────────────┼───────────────────┼─────────────┼────────────────────────────────────┼─────────────────────┤
│         1 │          9 │ false   │ 2022-06-06     │ 2022-03-01 │ Charles Smith 24   │ … │           5.0 │              NULL │ true        │ This 'newish' airport is compact,… │ 2026-06-20 12:45:

## Step 2: Column descriptions

One line per column. Meanings are inferred from the column name and the data, so correct them if you know better.

| Column | Type | Meaning |
|---|---|---|
| `review_id` | BIGINT | Review id |
| `airport_id` | BIGINT | Airport reviewed, joins to `raw_airports` |
| `verify` | BOOLEAN | Verified review |
| `date_submitted` | DATE | Date the review was posted |
| `date_visit` | DATE | Month of the airport visit (first day of month) |
| `customer_name` | VARCHAR | Reviewer name |
| `nationality` | VARCHAR | Reviewer country |
| `experience_at_airport` | VARCHAR | Purpose of the airport visit (departure, arrival, transit, ...) |
| `type_of_traveller` | VARCHAR | Trip type |
| `queuing_times`, `terminal_cleanliness`, `terminal_seating`, `terminal_signs`, `food_beverages`, `airport_shopping`, `airport_staff`, `wifi_connectivity` | DOUBLE | Rating 1 to 5 per aspect |
| `recommended` | BOOLEAN | Reviewer recommends the airport |
| `review` | VARCHAR | Free-text review |
| `updated_at` | TIMESTAMP | Load time |

## Step 3: Nulls & coverage

Null count and null % for every column, highest first. A high null % is real missing data, not something to fill.

In [93]:
null_check(T)

┌───────────────────────┬────────┬──────────┐
│          col          │ n_null │ pct_null │
│        varchar        │ int64  │  double  │
├───────────────────────┼────────┼──────────┤
│ wifi_connectivity     │  27729 │    56.01 │
│ food_beverages        │  24202 │    48.89 │
│ terminal_seating      │  21413 │    43.25 │
│ airport_staff         │  20009 │    40.42 │
│ terminal_signs        │  17919 │     36.2 │
│ date_visit            │  17322 │    34.99 │
│ type_of_traveller     │  17271 │    34.89 │
│ experience_at_airport │  17249 │    34.84 │
│ airport_shopping      │  14366 │    29.02 │
│ terminal_cleanliness  │   5306 │    10.72 │
│ queuing_times         │   5218 │    10.54 │
│ nationality           │   3107 │     6.28 │
│ airport_id            │      0 │      0.0 │
│ customer_name         │      0 │      0.0 │
│ date_submitted        │      0 │      0.0 │
│ recommended           │      0 │      0.0 │
│ review                │      1 │      0.0 │
│ review_id             │      0 │

## Step 4: Profile by column type

Overview first (one line per column), then the type-specific checks.

In [94]:
profile_overview(T)

┌───────────────────────┬───────────┬────────────┬─────────────────────┬─────────────────────┐
│          col          │ col_type  │ n_distinct │       min_val       │       max_val       │
│        varchar        │  varchar  │   int64    │       varchar       │       varchar       │
├───────────────────────┼───────────┼────────────┼─────────────────────┼─────────────────────┤
│ review_id             │ BIGINT    │      49505 │ 1                   │ 49505               │
│ airport_id            │ BIGINT    │       1004 │ 1                   │ 1004                │
│ verify                │ BOOLEAN   │          2 │ NULL                │ NULL                │
│ date_submitted        │ DATE      │       6171 │ 2002-07-30          │ 2026-06-20          │
│ date_visit            │ DATE      │        144 │ 2014-07-01          │ 2026-06-01          │
│ customer_name         │ VARCHAR   │      33296 │ 7                   │ 24                  │
│ nationality           │ VARCHAR   │       1845 │

### 4a. varchar

Distinct count, min/max length, most frequent value. Then the top values of categorical columns.

In [95]:
varchar_profile(T)

┌───────────────────────┬────────────┬─────────┬─────────┬──────────────────┐
│          col          │ n_distinct │ min_len │ max_len │  most_frequent   │
│        varchar        │   int64    │  int64  │  int64  │     varchar      │
├───────────────────────┼────────────┼─────────┼─────────┼──────────────────┤
│ customer_name         │      33296 │       7 │      24 │ Linda Roberts 16 │
│ nationality           │       1845 │       1 │     100 │ United Kingdom   │
│ experience_at_airport │          4 │       7 │      21 │ Departure Only   │
│ type_of_traveller     │          4 │       8 │      14 │ Couple Leisure   │
│ review                │      48818 │      15 │    5122 │ ✅ Trip Verified │
└───────────────────────┴────────────┴─────────┴─────────┴──────────────────┘



In [96]:
for col in ["experience_at_airport", "type_of_traveller"]:
    top_values(T, col)

┌───────────────────────┬───────┬────────┐
│         value         │   n   │  pct   │
│        varchar        │ int64 │ double │
├───────────────────────┼───────┼────────┤
│ NULL                  │ 17249 │  34.84 │
│ Departure Only        │ 12337 │  24.92 │
│ Arrival and Departure │ 11836 │  23.91 │
│ Transit               │  4091 │   8.26 │
│ Arrival Only          │  3992 │   8.06 │
└───────────────────────┴───────┴────────┘

┌────────────────┬───────┬────────┐
│     value      │   n   │  pct   │
│    varchar     │ int64 │ double │
├────────────────┼───────┼────────┤
│ NULL           │ 17271 │  34.89 │
│ Couple Leisure │  9637 │  19.47 │
│ Solo Leisure   │  9370 │  18.93 │
│ Family Leisure │  7170 │  14.48 │
│ Business       │  6057 │  12.24 │
└────────────────┴───────┴────────┘



In [97]:
top_values(T, "nationality", 10)

┌────────────────┬───────┬────────┐
│     value      │   n   │  pct   │
│    varchar     │ int64 │ double │
├────────────────┼───────┼────────┤
│ United Kingdom │ 15792 │   31.9 │
│ United States  │  8863 │   17.9 │
│ NULL           │  3107 │   6.28 │
│ Australia      │  3075 │   6.21 │
│ Canada         │  1945 │   3.93 │
│ Germany        │  1107 │   2.24 │
│ Netherlands    │   857 │   1.73 │
│ Ireland        │   852 │   1.72 │
│ India          │   782 │   1.58 │
│ France         │   688 │   1.39 │
└────────────────┴───────┴────────┘
  10 rows               3 columns



### 4b. numeric

Min, max, avg, median. For rating columns the histogram should show whole numbers 1 to 5 only.

In [98]:
numeric_stats(T)

┌──────────────────────┬────────────┬─────────┬─────────┬─────────┬────────────┐
│         col          │ n_distinct │ min_val │ max_val │ avg_val │ median_val │
│       varchar        │   int64    │ double  │ double  │ double  │   double   │
├──────────────────────┼────────────┼─────────┼─────────┼─────────┼────────────┤
│ review_id            │      49505 │     1.0 │ 49505.0 │ 24753.0 │    24753.0 │
│ airport_id           │       1004 │     1.0 │  1004.0 │  470.86 │      521.0 │
│ queuing_times        │          5 │     1.0 │     5.0 │    2.38 │        2.0 │
│ terminal_cleanliness │          5 │     1.0 │     5.0 │    3.04 │        3.0 │
│ terminal_seating     │          5 │     1.0 │     5.0 │    2.38 │        2.0 │
│ terminal_signs       │          5 │     1.0 │     5.0 │    2.73 │        3.0 │
│ food_beverages       │          5 │     1.0 │     5.0 │    2.37 │        2.0 │
│ airport_shopping     │          5 │     1.0 │     5.0 │    2.59 │        3.0 │
│ airport_staff        │    

In [99]:
numeric_hist(T)

┌──────────────────────┬────────┬────────┬────────┬────────┬────────┐
│         col          │  1.0   │  2.0   │  3.0   │  4.0   │  5.0   │
│       varchar        │ int128 │ int128 │ int128 │ int128 │ int128 │
├──────────────────────┼────────┼────────┼────────┼────────┼────────┤
│ airport_shopping     │  10736 │   6352 │   8578 │   5369 │   4104 │
│ airport_staff        │  15049 │   4037 │   3760 │   3310 │   3340 │
│ food_beverages       │   9401 │   5205 │   4944 │   3430 │   2323 │
│ queuing_times        │  20413 │   5000 │   6849 │   5584 │   6441 │
│ terminal_cleanliness │   8713 │   7021 │  10936 │   8784 │   8745 │
│ terminal_seating     │  10959 │   5294 │   4979 │   3949 │   2911 │
│ terminal_signs       │   9045 │   5775 │   6416 │   5420 │   4930 │
│ wifi_connectivity    │   8029 │   3319 │   4012 │   3325 │   3091 │
└──────────────────────┴────────┴────────┴────────┴────────┴────────┘



### 4c. date

Min, max, rows before 1990 (placeholder dates such as 1970-01-01) and future dates.

In [100]:
date_stats(T)

┌────────────────┬─────────────────────┬─────────────────────┬───────────────┬──────────┬────────────┐
│      col       │       min_val       │       max_val       │ n_before_1990 │ n_future │ n_distinct │
│    varchar     │       varchar       │       varchar       │     int64     │  int64   │   int64    │
├────────────────┼─────────────────────┼─────────────────────┼───────────────┼──────────┼────────────┤
│ date_submitted │ 2002-07-30          │ 2026-06-20          │             0 │        0 │       6171 │
│ date_visit     │ 2014-07-01          │ 2026-06-01          │             0 │        0 │        144 │
│ updated_at     │ 2026-06-20 12:45:42 │ 2026-06-20 12:45:42 │             0 │        0 │          1 │
└────────────────┴─────────────────────┴─────────────────────┴───────────────┴──────────┴────────────┘



In [101]:
con.sql(f"select year(date_submitted) y, count(*) n from {T} group by 1 order by 1")

┌───────┬───────┐
│   y   │   n   │
│ int64 │ int64 │
├───────┼───────┤
│  2002 │    24 │
│  2003 │   171 │
│  2004 │   360 │
│  2005 │   611 │
│  2006 │   700 │
│  2007 │   706 │
│  2008 │  1149 │
│  2009 │  1335 │
│  2010 │  1420 │
│  2011 │  1995 │
│    ·  │    ·  │
│    ·  │    ·  │
│    ·  │    ·  │
│  2017 │  4238 │
│  2018 │  4895 │
│  2019 │  4836 │
│  2020 │  1219 │
│  2021 │  1318 │
│  2022 │  2581 │
│  2023 │  2474 │
│  2024 │  2075 │
│  2025 │  1236 │
│  2026 │   414 │
└───────┴───────┘
     25 rows   
   (20 shown)   

### 4d. boolean

Distribution of True / False / null.

In [102]:
bool_dist(T)

┌─────────────┬────────┬─────────┬────────┐
│     col     │ n_true │ n_false │ n_null │
│   varchar   │ int64  │  int64  │ int64  │
├─────────────┼────────┼─────────┼────────┤
│ verify      │  10669 │   38836 │      0 │
│ recommended │  11681 │   37824 │      0 │
└─────────────┴────────┴─────────┴────────┘



## Step 5: Caveat hunting

Keys and duplicates first, then whitespace, then table-specific checks. Anything found here goes in the summary with a planned fix for dbt staging.

In [103]:
key_checks(T, 'review_id', {'airport_id': 'raw_airports'})

┌────────────────┬─────────────────────┬───────────────────┐
│ duplicate_keys │ duplicate_full_rows │ orphan_airport_id │
│     int64      │        int64        │       int64       │
├────────────────┼─────────────────────┼───────────────────┤
│              0 │                   0 │                 0 │
└────────────────┴─────────────────────┴───────────────────┘



In [104]:
whitespace_check(T)

┌───────────────────────┬──────────────┬────────────┐
│          col          │ n_whitespace │ n_one_char │
│        varchar        │    int64     │   int64    │
├───────────────────────┼──────────────┼────────────┤
│ customer_name         │            0 │          0 │
│ nationality           │            0 │         16 │
│ experience_at_airport │            0 │          0 │
│ type_of_traveller     │            0 │          0 │
│ review                │            0 │          0 │
└───────────────────────┴──────────────┴────────────┘



In [105]:
# date_visit after date_submitted would be impossible
con.sql(f"select count(*) as visit_after_submitted from {T} where date_visit > date_submitted")

┌───────────────────────┐
│ visit_after_submitted │
│         int64         │
├───────────────────────┤
│                     0 │
└───────────────────────┘

In [106]:
# experience_at_airport / type_of_traveller written with different case or spelling
con.sql(f"""
    select lower(trim(experience_at_airport)) k, list(distinct experience_at_airport) variants
    from {T} group by 1 having count(distinct experience_at_airport) > 1
""").show(max_width=200)

┌─────────┬───────────┐
│    k    │ variants  │
│ varchar │ varchar[] │
└─────────┴───────────┘
        0 rows       



In [107]:
# customer_name ending with a number, and nationality holding digits or punctuation (same patterns as the other fact tables)
con.sql(f"""
    select count(*) filter (where regexp_matches(customer_name, ' [0-9]+$'))      as names_with_trailing_number,
           count(*) filter (where regexp_matches(nationality, '[0-9]'))            as nationality_with_digit
    from {T}
""")

┌────────────────────────────┬────────────────────────┐
│ names_with_trailing_number │ nationality_with_digit │
│           int64            │         int64          │
├────────────────────────────┼────────────────────────┤
│                      47722 │                    326 │
└────────────────────────────┴────────────────────────┘

In [108]:
# nationality values that are not a plain country name: digits or '.' (e.g. 'flt. 93' is a flight number)
con.sql(f"""
    select nationality, count(*) n from {T}
    where regexp_matches(nationality, '[0-9.]')
    group by 1 order by n desc limit 20
""")

┌───────────────────────────────────────────────────────────────────────────────────────────────┬───────┐
│                                          nationality                                          │   n   │
│                                            varchar                                            │ int64 │
├───────────────────────────────────────────────────────────────────────────────────────────────┼───────┤
│ 1                                                                                             │     3 │
│ about 15 of us                                                                                │     2 │
│ Terminal 2 only                                                                               │     2 │
│ Terminal 2                                                                                    │     2 │
│ at least 2 hours waiting time                                                                 │     2 │
│ 1315                                        

In [109]:
# how many nationality values are NOT a plain country name
con.sql(f"""
    select count(*) filter (where length(nationality) > 25)               as n_longer_than_25_chars,
           count(*) filter (where regexp_matches(nationality, '[0-9.]'))  as n_with_digit_or_dot,
           count(*) filter (where length(nationality) <= 2)               as n_two_chars_or_less,
           count(distinct nationality)                                    as n_distinct
    from {T}
""").show(max_width=200)

┌────────────────────────┬─────────────────────┬─────────────────────┬────────────┐
│ n_longer_than_25_chars │ n_with_digit_or_dot │ n_two_chars_or_less │ n_distinct │
│         int64          │        int64        │        int64        │   int64    │
├────────────────────────┼─────────────────────┼─────────────────────┼────────────┤
│                    776 │                 378 │                  30 │       1845 │
└────────────────────────┴─────────────────────┴─────────────────────┴────────────┘



In [110]:
dup_text(T, "review")

┌────────────────────────────┐
│ rows_with_duplicate_review │
│           int64            │
├────────────────────────────┤
│                       1371 │
└────────────────────────────┘



In [111]:
# same entity + same review text submitted more than once (true duplicate reviews)
con.sql(f"""
    select count(*) as groups_with_same_entity_and_text
    from (select 1 from {T} group by airport_id, review having count(*) > 1)
""")

┌──────────────────────────────────┐
│ groups_with_same_entity_and_text │
│              int64               │
├──────────────────────────────────┤
│                              114 │
└──────────────────────────────────┘

In [112]:
# review text that is only a verification marker ('Not Verified', 'Trip Verified' with a check-mark prefix) or empty
con.sql(f"""
    select count(*) filter (where regexp_matches(review, '^[^A-Za-z]*(Not Verified|Trip Verified)$')) as n_review_is_marker,
           count(*) filter (where review is null)                                                 as n_review_null
    from {T}
""")

┌────────────────────┬───────────────┐
│ n_review_is_marker │ n_review_null │
│       int64        │     int64     │
├────────────────────┼───────────────┤
│                  3 │             1 │
└────────────────────┴───────────────┘

## Step 6: Summary of `raw_airport_reviews`

**Purpose:** Fact table of airport reviews, one row per review. Joins to `raw_airports` on `airport_id`.

**Shape:** 49,505 rows x 20 columns.

| Column group | Columns | DuckDB type | Meaning | Null % | Profile |
|---|---|---|---|---|---|
| Keys | `review_id`, `airport_id` | BIGINT | Review id and airport | 0% | `review_id` 1-49505, all unique; 1,004 distinct airports (all of `raw_airports`) |
| Flags | `verify`, `recommended` | BOOLEAN | Verified review; recommends airport | 0% | verify: 10,669 True / 38,836 False; recommended: 11,681 True / 37,824 False |
| Dates | `date_submitted`, `date_visit` | DATE | Review post date; visit month | 0% / 35.0% | submitted 2002-07-30 to 2026-06-20; visit 2014-07-01 to 2026-06-01 |
| Reviewer | `customer_name`, `nationality` | VARCHAR | Reviewer name and country | 0% / 6.3% | 33,296 / 1,845 distinct; nationality max length 100 |
| Trip | `experience_at_airport`, `type_of_traveller` | VARCHAR | Departure / arrival / transit; trip type | 34.8% / 34.9% | 4 clean values each |
| Ratings | `queuing_times`, `terminal_cleanliness`, `terminal_seating`, `terminal_signs`, `food_beverages`, `airport_shopping`, `airport_staff`, `wifi_connectivity` | DOUBLE | Rating 1-5 | 10.5% to 56.0% | integers 1-5 only |
| Text | `review` | VARCHAR | Free-text review | 1 null row | length 15-5,122 |
| Metadata | `updated_at` | TIMESTAMP | Load time | 0% | one value for all rows |

**Caveat checks:**
- Duplicate `review_id`: 0. Orphaned `airport_id`: 0. Full-row duplicates: 0.
- Leading/trailing whitespace: 0 in every text column.
- No future dates, no placeholder dates before 1990, `date_visit` never later than `date_submitted`.
- Ratings hold whole numbers 1-5 only. `type_of_traveller` and `experience_at_airport` have no spelling or case variants.
- **`nationality` is polluted with free text.** 776 rows are longer than 25 characters, 378 contain a digit or a dot (e.g. `Terminal 2`, `Gate 2`, `45min`, `$3.30 Macau dollars`), 30 are two characters or fewer (`!`, `?`, `s`, `1`). Review text leaked into the wrong column; 1,845 distinct values for what should be about 200 countries.
- **`customer_name` ends with a number in 47,722 of 49,505 rows** (e.g. `Charles Smith 24`). The suffix is not part of a real name.
- **Duplicate `review` text:** 1,371 rows share their text with another row (685 groups, up to 3 copies); 114 groups are the same airport with the same text, which are true duplicate reviews.
- **Review text edge cases:** 1 null review; 3 rows contain only the marker `Trip Verified` (prefixed by a check-mark character in the data).
- `updated_at` carries no information.
- High nulls in the ratings, `date_visit`, `type_of_traveller` and `experience_at_airport` are real missing data (not every reviewer fills every field).

**Conclusion:** Keys and the join to `raw_airports` are sound. In staging: clean `nationality` (keep only values that match a country, set the rest to null), strip the numeric suffix from `customer_name`, decide how to treat duplicate review text (at least the 114 same-airport duplicates), and drop `updated_at`. Do not fill the nulls.

# Table 6: `raw_airline_reviews`

**Role:** Fact table of airline (flight) reviews, one row per review. The largest table. Joins to `raw_airlines` on `airline_id`.

**Steps:** 1 Shape & schema, 2 Column descriptions, 3 Nulls & coverage, 4 Profile by type (4a varchar, 4b numeric, 4c date, 4d boolean), 5 Caveat hunting, 6 Summary.

In [113]:
T = "raw_airline_reviews"

## Step 1: Shape & schema

How many rows and columns, what types DuckDB inferred, and what the data looks like.

In [114]:
con.sql(f"describe {T}").show(max_rows=40, max_width=200)

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ review_id              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ airline_id             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ verify                 │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_submitted         │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ date_flown             │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_name          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ nationality            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ type_of_traveller      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ seat_type              │ V

In [115]:
con.sql(f"select count(*) as n_rows from {T}")

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│ 156323 │
└────────┘

In [116]:
con.sql(f"select * from {T} limit 10").show(max_width=200)

┌───────────┬────────────┬─────────┬────────────────┬────────────┬───────────────────┬───┬───────────────────────┬─────────────────┬─────────────┬───────────────────────────────┬─────────────────────┐
│ review_id │ airline_id │ verify  │ date_submitted │ date_flown │   customer_name   │ … │ wifi_and_connectivity │ value_for_money │ recommended │            review             │     updated_at      │
│   int64   │   int64    │ boolean │      date      │    date    │      varchar      │ … │        double         │     double      │   boolean   │            varchar            │      timestamp      │
├───────────┼────────────┼─────────┼────────────────┼────────────┼───────────────────┼───┼───────────────────────┼─────────────────┼─────────────┼───────────────────────────────┼─────────────────────┤
│         1 │        313 │ false   │ 2002-01-06     │ NULL       │ Joshua Torres 6   │ … │                  NULL │            NULL │ false       │ I flew Lan Peru on a domesti… │ 2026-04-03 14:30:

## Step 2: Column descriptions

One line per column. Meanings are inferred from the column name and the data, so correct them if you know better.

| Column | Type | Meaning |
|---|---|---|
| `review_id` | BIGINT | Review id |
| `airline_id` | BIGINT | Airline reviewed, joins to `raw_airlines` |
| `verify` | BOOLEAN | Verified review |
| `date_submitted` | DATE | Date the review was posted |
| `date_flown` | DATE | Month of the flight (first day of month) |
| `customer_name` | VARCHAR | Reviewer name |
| `nationality` | VARCHAR | Reviewer country |
| `type_of_traveller` | VARCHAR | Trip type |
| `seat_type` | VARCHAR | Cabin class |
| `aircraft` | VARCHAR | Aircraft model flown |
| `origin_city`, `origin_airport` | VARCHAR | Where the flight started |
| `destination_city`, `destination_airport` | VARCHAR | Where the flight ended |
| `transit_city`, `transit_airport` | VARCHAR | Stop-over, if any |
| `seat_comfort`, `cabin_staff_service`, `food_and_beverages`, `inflight_entertainment`, `ground_service`, `wifi_and_connectivity`, `value_for_money` | DOUBLE | Rating 1 to 5 per aspect |
| `recommended` | BOOLEAN | Reviewer recommends the airline |
| `review` | VARCHAR | Free-text review |
| `updated_at` | TIMESTAMP | Load time |

## Step 3: Nulls & coverage

Null count and null % for every column, highest first. A high null % is real missing data, not something to fill.

In [117]:
null_check(T)

┌────────────────────────┬────────┬──────────┐
│          col           │ n_null │ pct_null │
│        varchar         │ int64  │  double  │
├────────────────────────┼────────┼──────────┤
│ transit_airport        │ 134817 │    86.24 │
│ transit_city           │ 126218 │    80.74 │
│ wifi_and_connectivity  │ 115123 │    73.64 │
│ aircraft               │ 115094 │    73.63 │
│ destination_airport    │  82725 │    52.92 │
│ origin_airport         │  80300 │    51.37 │
│ inflight_entertainment │  65404 │    41.84 │
│ food_and_beverages     │  44381 │    28.39 │
│ ground_service         │  43556 │    27.86 │
│ destination_city       │  40154 │    25.69 │
│ origin_city            │  39913 │    25.53 │
│ date_flown             │  39607 │    25.34 │
│ type_of_traveller      │  39446 │    25.23 │
│ cabin_staff_service    │  16593 │    10.61 │
│ seat_comfort           │  16225 │    10.38 │
│ seat_type              │   3024 │     1.93 │
│ value_for_money        │   2417 │     1.55 │
│ nationality

## Step 4: Profile by column type

Overview first (one line per column), then the type-specific checks.

In [118]:
profile_overview(T)

┌────────────────────────┬───────────┬────────────┬─────────────────────┬─────────────────────┐
│          col           │ col_type  │ n_distinct │       min_val       │       max_val       │
│        varchar         │  varchar  │   int64    │       varchar       │       varchar       │
├────────────────────────┼───────────┼────────────┼─────────────────────┼─────────────────────┤
│ review_id              │ BIGINT    │     156323 │ 1                   │ 156323              │
│ airline_id             │ BIGINT    │        580 │ 1                   │ 594                 │
│ verify                 │ BOOLEAN   │          2 │ NULL                │ NULL                │
│ date_submitted         │ DATE      │       6257 │ 2002-01-06          │ 2026-04-04          │
│ date_flown             │ DATE      │        146 │ 2012-04-01          │ 2026-04-01          │
│ customer_name          │ VARCHAR   │     104705 │ 7                   │ 24                  │
│ nationality            │ VARCHAR   │  

### 4a. varchar

Distinct count, min/max length, most frequent value. Then the top values of categorical columns.

In [119]:
varchar_profile(T)

┌─────────────────────┬────────────┬─────────┬─────────┬───────────────┐
│         col         │ n_distinct │ min_len │ max_len │ most_frequent │
│       varchar       │   int64    │  int64  │  int64  │    varchar    │
├─────────────────────┼────────────┼─────────┼─────────┼───────────────┤
│ customer_name       │     104705 │       7 │      24 │ Susan Allen 2 │
│ nationality         │        747 │       1 │     100 │ United States │
│ type_of_traveller   │          4 │       8 │      14 │ Solo Leisure  │
│ seat_type           │          4 │      11 │      15 │ Economy Class │
│ aircraft            │        226 │       6 │      31 │ Airbus A320   │
│ origin_city         │       3784 │       1 │      45 │ London        │
│ origin_airport      │        195 │       3 │       3 │ LHR           │
│ destination_city    │       4207 │       1 │      34 │ London        │
│ destination_airport │        195 │       3 │       3 │ LHR           │
│ transit_city        │       1206 │       2 │     

In [120]:
for col in ["type_of_traveller", "seat_type"]:
    top_values(T, col)

┌────────────────┬───────┬────────┐
│     value      │   n   │  pct   │
│    varchar     │ int64 │ double │
├────────────────┼───────┼────────┤
│ Solo Leisure   │ 42340 │  27.08 │
│ NULL           │ 39446 │  25.23 │
│ Couple Leisure │ 30371 │  19.43 │
│ Family Leisure │ 26322 │  16.84 │
│ Business       │ 17844 │  11.41 │
└────────────────┴───────┴────────┘

┌─────────────────┬────────┬────────┐
│      value      │   n    │  pct   │
│     varchar     │ int64  │ double │
├─────────────────┼────────┼────────┤
│ Economy Class   │ 124792 │  79.83 │
│ Business Class  │  19199 │  12.28 │
│ Premium Economy │   6099 │    3.9 │
│ First Class     │   3209 │   2.05 │
│ NULL            │   3024 │   1.93 │
└─────────────────┴────────┴────────┘



In [121]:
top_values(T, "aircraft", 15)

┌──────────────────┬────────┬────────┐
│      value       │   n    │  pct   │
│     varchar      │ int64  │ double │
├──────────────────┼────────┼────────┤
│ NULL             │ 115094 │  73.63 │
│ Airbus A320      │   6533 │   4.18 │
│ Airbus A330      │   4426 │   2.83 │
│ Boeing 737-800   │   3252 │   2.08 │
│ Boeing 787       │   2576 │   1.65 │
│ Boeing 737       │   2470 │   1.58 │
│ Boeing 777       │   2345 │    1.5 │
│ Airbus A380      │   2269 │   1.45 │
│ Airbus A321      │   2102 │   1.34 │
│ Boeing 777-300ER │   1550 │   0.99 │
│ Airbus A319      │   1481 │   0.95 │
│ Airbus A350      │   1465 │   0.94 │
│ Boeing 777-300   │   1122 │   0.72 │
│ Boeing 787-9     │    791 │   0.51 │
│ Boeing 777-200   │    745 │   0.48 │
└──────────────────┴────────┴────────┘
  15 rows                  3 columns



In [122]:
top_values(T, "nationality", 10)

┌────────────────┬───────┬────────┐
│     value      │   n   │  pct   │
│    varchar     │ int64 │ double │
├────────────────┼───────┼────────┤
│ United States  │ 51232 │  32.77 │
│ United Kingdom │ 24949 │  15.96 │
│ Australia      │ 13424 │   8.59 │
│ Canada         │ 12769 │   8.17 │
│ Germany        │  4243 │   2.71 │
│ India          │  3705 │   2.37 │
│ Singapore      │  2421 │   1.55 │
│ Netherlands    │  2300 │   1.47 │
│ France         │  2125 │   1.36 │
│ New Zealand    │  1781 │   1.14 │
└────────────────┴───────┴────────┘
  10 rows               3 columns



In [123]:
top_values(T, "origin_city", 10)

┌───────────────┬───────┬────────┐
│     value     │   n   │  pct   │
│    varchar    │ int64 │ double │
├───────────────┼───────┼────────┤
│ NULL          │ 39913 │  25.53 │
│ London        │  7553 │   4.83 │
│ New York      │  4030 │   2.58 │
│ Toronto       │  2487 │   1.59 │
│ Los Angeles   │  2430 │   1.55 │
│ Bangkok       │  2154 │   1.38 │
│ Sydney        │  1863 │   1.19 │
│ Singapore     │  1820 │   1.16 │
│ Chicago       │  1752 │   1.12 │
│ San Francisco │  1524 │   0.97 │
└───────────────┴───────┴────────┘
  10 rows              3 columns



### 4b. numeric

Min, max, avg, median. For rating columns the histogram should show whole numbers 1 to 5 only.

In [124]:
numeric_stats(T)

┌────────────────────────┬────────────┬─────────┬──────────┬─────────┬────────────┐
│          col           │ n_distinct │ min_val │ max_val  │ avg_val │ median_val │
│        varchar         │   int64    │ double  │  double  │ double  │   double   │
├────────────────────────┼────────────┼─────────┼──────────┼─────────┼────────────┤
│ review_id              │     156323 │     1.0 │ 156323.0 │ 78162.0 │    78162.0 │
│ airline_id             │        580 │     1.0 │    594.0 │  310.28 │      300.0 │
│ seat_comfort           │          5 │     1.0 │      5.0 │    2.71 │        3.0 │
│ cabin_staff_service    │          5 │     1.0 │      5.0 │    2.97 │        3.0 │
│ food_and_beverages     │          5 │     1.0 │      5.0 │    2.69 │        3.0 │
│ inflight_entertainment │          5 │     1.0 │      5.0 │     2.6 │        3.0 │
│ ground_service         │          5 │     1.0 │      5.0 │    2.37 │        1.0 │
│ wifi_and_connectivity  │          5 │     1.0 │      5.0 │    1.98 │      

In [125]:
numeric_hist(T)

┌────────────────────────┬────────┬────────┬────────┬────────┬────────┐
│          col           │  1.0   │  2.0   │  3.0   │  4.0   │  5.0   │
│        varchar         │ int128 │ int128 │ int128 │ int128 │ int128 │
├────────────────────────┼────────┼────────┼────────┼────────┼────────┤
│ cabin_staff_service    │  42007 │  17137 │  20861 │  23161 │  36564 │
│ food_and_beverages     │  38071 │  15281 │  19850 │  20376 │  18364 │
│ ground_service         │  56942 │  10039 │  11741 │  14694 │  19351 │
│ inflight_entertainment │  34603 │  10718 │  15595 │  16000 │  14003 │
│ seat_comfort           │  44203 │  20528 │  27308 │  27492 │  20567 │
│ value_for_money        │  64550 │  17870 │  17116 │  24757 │  29613 │
│ wifi_and_connectivity  │  25333 │   3720 │   4193 │   3553 │   4401 │
└────────────────────────┴────────┴────────┴────────┴────────┴────────┘



### 4c. date

Min, max, rows before 1990 (placeholder dates such as 1970-01-01) and future dates.

In [126]:
date_stats(T)

┌────────────────┬─────────────────────┬─────────────────────┬───────────────┬──────────┬────────────┐
│      col       │       min_val       │       max_val       │ n_before_1990 │ n_future │ n_distinct │
│    varchar     │       varchar       │       varchar       │     int64     │  int64   │   int64    │
├────────────────┼─────────────────────┼─────────────────────┼───────────────┼──────────┼────────────┤
│ date_submitted │ 2002-01-06          │ 2026-04-04          │             0 │        0 │       6257 │
│ date_flown     │ 2012-04-01          │ 2026-04-01          │             0 │        0 │        146 │
│ updated_at     │ 2026-04-03 14:30:19 │ 2026-04-04 14:04:45 │             0 │        0 │         61 │
└────────────────┴─────────────────────┴─────────────────────┴───────────────┴──────────┴────────────┘



In [127]:
con.sql(f"select year(date_submitted) y, count(*) n from {T} group by 1 order by 1")

┌───────┬───────┐
│   y   │   n   │
│ int64 │ int64 │
├───────┼───────┤
│  2002 │    16 │
│  2003 │    51 │
│  2004 │   137 │
│  2005 │   188 │
│  2006 │   238 │
│  2007 │   361 │
│  2008 │   426 │
│  2009 │   592 │
│  2010 │  1488 │
│  2011 │  3081 │
│    ·  │    ·  │
│    ·  │    ·  │
│    ·  │    ·  │
│  2017 │ 12169 │
│  2018 │ 14935 │
│  2019 │ 16251 │
│  2020 │  6156 │
│  2021 │  6780 │
│  2022 │ 12896 │
│  2023 │ 12781 │
│  2024 │  7466 │
│  2025 │  4404 │
│  2026 │   801 │
└───────┴───────┘
     25 rows   
   (20 shown)   

### 4d. boolean

Distribution of True / False / null.

In [128]:
bool_dist(T)

┌─────────────┬────────┬─────────┬────────┐
│     col     │ n_true │ n_false │ n_null │
│   varchar   │ int64  │  int64  │ int64  │
├─────────────┼────────┼─────────┼────────┤
│ verify      │  61970 │   94353 │      0 │
│ recommended │  58562 │   97761 │      0 │
└─────────────┴────────┴─────────┴────────┘



## Step 5: Caveat hunting

Keys and duplicates first, then whitespace, then table-specific checks. Anything found here goes in the summary with a planned fix for dbt staging.

In [129]:
key_checks(T, 'review_id', {'airline_id': 'raw_airlines'})

┌────────────────┬─────────────────────┬───────────────────┐
│ duplicate_keys │ duplicate_full_rows │ orphan_airline_id │
│     int64      │        int64        │       int64       │
├────────────────┼─────────────────────┼───────────────────┤
│              0 │                   0 │                 0 │
└────────────────┴─────────────────────┴───────────────────┘



In [130]:
whitespace_check(T)

┌─────────────────────┬──────────────┬────────────┐
│         col         │ n_whitespace │ n_one_char │
│       varchar       │    int64     │   int64    │
├─────────────────────┼──────────────┼────────────┤
│ customer_name       │            0 │          0 │
│ nationality         │            0 │          5 │
│ type_of_traveller   │            0 │          0 │
│ seat_type           │            0 │          0 │
│ aircraft            │            0 │          0 │
│ origin_city         │            0 │          1 │
│ origin_airport      │            0 │          0 │
│ destination_city    │            0 │          1 │
│ destination_airport │            0 │          0 │
│ transit_city        │            0 │          0 │
│ transit_airport     │            0 │          0 │
│ review              │            0 │          0 │
└─────────────────────┴──────────────┴────────────┘
  12 rows                               3 columns



In [131]:
# date_flown after date_submitted would be impossible
con.sql(f"select count(*) as flown_after_submitted from {T} where date_flown > date_submitted")

┌───────────────────────┐
│ flown_after_submitted │
│         int64         │
├───────────────────────┤
│                    41 │
└───────────────────────┘

In [132]:
# category columns written with different case or spelling: same value, several variants
con.sql(f"""
    select 'seat_type' as col, lower(trim(seat_type)) k, list(distinct seat_type) variants
    from {T} group by 2 having count(distinct seat_type) > 1
    union all
    select 'type_of_traveller', lower(trim(type_of_traveller)), list(distinct type_of_traveller)
    from {T} group by 2 having count(distinct type_of_traveller) > 1
""").show(max_width=200)

┌─────────┬─────────┬───────────┐
│   col   │    k    │ variants  │
│ varchar │ varchar │ varchar[] │
└─────────┴─────────┴───────────┘
             0 rows            



In [133]:
# route columns: origin = destination, and transit filled while origin or destination is empty
con.sql(f"""
    select count(*) filter (where origin_city = destination_city)                              as same_origin_destination_city,
           count(*) filter (where origin_airport = destination_airport)                        as same_origin_destination_airport,
           count(*) filter (where transit_city is not null and origin_city is null)            as transit_without_origin
    from {T}
""").show(max_width=200)

┌──────────────────────────────┬─────────────────────────────────┬────────────────────────┐
│ same_origin_destination_city │ same_origin_destination_airport │ transit_without_origin │
│            int64             │              int64              │         int64          │
├──────────────────────────────┼─────────────────────────────────┼────────────────────────┤
│                           35 │                              20 │                      0 │
└──────────────────────────────┴─────────────────────────────────┴────────────────────────┘



In [134]:
# same airline + same review text submitted more than once (look at the first few review_ids, e.g. 1 and 2)
con.sql(f"""
    select airline_id, review_id, date_submitted, left(review, 60) as review_start
    from {T}
    where review_id in (1, 2, 3)
""").show(max_width=200)

┌────────────┬───────────┬────────────────┬──────────────────────────────────────────────────────────────┐
│ airline_id │ review_id │ date_submitted │                         review_start                         │
│   int64    │   int64   │      date      │                           varchar                            │
├────────────┼───────────┼────────────────┼──────────────────────────────────────────────────────────────┤
│        313 │         1 │ 2002-01-06     │ I flew Lan Peru on a domestic flight from Lima to Cusco in A │
│        313 │         2 │ 2002-01-06     │ I flew Lan Peru on a domestic flight from Lima to Cusco in A │
│        466 │         3 │ 2002-02-14     │ Skymark is a low-fare airline in Japan. I flow from Tokyo Ha │
└────────────┴───────────┴────────────────┴──────────────────────────────────────────────────────────────┘



In [135]:
# customer_name ending with a number, and nationality holding digits or punctuation (same patterns as the other fact tables)
con.sql(f"""
    select count(*) filter (where regexp_matches(customer_name, ' [0-9]+$'))      as names_with_trailing_number,
           count(*) filter (where regexp_matches(nationality, '[0-9]'))            as nationality_with_digit
    from {T}
""")

┌────────────────────────────┬────────────────────────┐
│ names_with_trailing_number │ nationality_with_digit │
│           int64            │         int64          │
├────────────────────────────┼────────────────────────┤
│                     150886 │                    167 │
└────────────────────────────┴────────────────────────┘

In [136]:
# nationality values that are not a plain country name: digits or '.' (e.g. 'flt. 93' is a flight number)
con.sql(f"""
    select nationality, count(*) n from {T}
    where regexp_matches(nationality, '[0-9.]')
    group by 1 order by n desc limit 20
""")

┌──────────────────────────────────────────────────────────────────────────────┬───────┐
│                                 nationality                                  │   n   │
│                                   varchar                                    │ int64 │
├──────────────────────────────────────────────────────────────────────────────┼───────┤
│ B747                                                                         │     3 │
│ 24 Jan                                                                       │     2 │
│ 4 adults & 2 kids                                                            │     2 │
│ 1am                                                                          │     2 │
│ flt. 93                                                                      │     2 │
│ there are now 2 per day                                                      │     2 │
│ thin around 40 about 5'5' dark hair to his shoulders                         │     2 │
│ USD 500 to 700 for 

In [137]:
# how many nationality values are NOT a plain country name
con.sql(f"""
    select count(*) filter (where length(nationality) > 25)               as n_longer_than_25_chars,
           count(*) filter (where regexp_matches(nationality, '[0-9.]'))  as n_with_digit_or_dot,
           count(*) filter (where length(nationality) <= 2)               as n_two_chars_or_less,
           count(distinct nationality)                                    as n_distinct
    from {T}
""").show(max_width=200)

┌────────────────────────┬─────────────────────┬─────────────────────┬────────────┐
│ n_longer_than_25_chars │ n_with_digit_or_dot │ n_two_chars_or_less │ n_distinct │
│         int64          │        int64        │        int64        │   int64    │
├────────────────────────┼─────────────────────┼─────────────────────┼────────────┤
│                    238 │                 176 │                   9 │        747 │
└────────────────────────┴─────────────────────┴─────────────────────┴────────────┘



In [138]:
dup_text(T, "review")

┌────────────────────────────┐
│ rows_with_duplicate_review │
│           int64            │
├────────────────────────────┤
│                      13958 │
└────────────────────────────┘



In [139]:
# same entity + same review text submitted more than once (true duplicate reviews)
con.sql(f"""
    select count(*) as groups_with_same_entity_and_text
    from (select 1 from {T} group by airline_id, review having count(*) > 1)
""")

┌──────────────────────────────────┐
│ groups_with_same_entity_and_text │
│              int64               │
├──────────────────────────────────┤
│                              261 │
└──────────────────────────────────┘

In [140]:
# review text that is only a verification marker ('Not Verified', 'Trip Verified' with a check-mark prefix) or empty
con.sql(f"""
    select count(*) filter (where regexp_matches(review, '^[^A-Za-z]*(Not Verified|Trip Verified)$')) as n_review_is_marker,
           count(*) filter (where review is null)                                                 as n_review_null
    from {T}
""")

┌────────────────────┬───────────────┐
│ n_review_is_marker │ n_review_null │
│       int64        │     int64     │
├────────────────────┼───────────────┤
│                  6 │             0 │
└────────────────────┴───────────────┘

## Step 6: Summary of `raw_airline_reviews`

**Purpose:** Fact table of airline (flight) reviews, one row per review. The largest table. Joins to `raw_airlines` on `airline_id`.

**Shape:** 156,323 rows x 26 columns.

| Column group | Columns | DuckDB type | Meaning | Null % | Profile |
|---|---|---|---|---|---|
| Keys | `review_id`, `airline_id` | BIGINT | Review id and airline | 0% | `review_id` 1-156323, all unique; 580 distinct airlines |
| Flags | `verify`, `recommended` | BOOLEAN | Verified review; recommends airline | 0% | verify: 61,970 True / 94,353 False; recommended: 58,562 True / 97,761 False |
| Dates | `date_submitted`, `date_flown` | DATE | Review post date; flight month | 0% / 25.3% | submitted 2002-01-06 to 2026-04-04; flown 2012-04-01 to 2026-04-01 |
| Reviewer | `customer_name`, `nationality` | VARCHAR | Reviewer name and country | 0% / 0.7% | 104,705 / 747 distinct; nationality max length 100 |
| Trip | `type_of_traveller`, `seat_type` | VARCHAR | Trip type; cabin class | 25.2% / 1.9% | 4 clean values each |
| Aircraft | `aircraft` | VARCHAR | Aircraft model | 73.6% | 226 distinct, free text (`Boeing 737`, `Boeing 737-800`, ...) |
| Route | `origin_city`, `origin_airport`, `destination_city`, `destination_airport`, `transit_city`, `transit_airport` | VARCHAR | Route of the flight; airports are 3-letter IATA codes | 25.5% / 51.4% / 25.7% / 52.9% / 80.7% / 86.2% | 3,784 / 195 / 4,207 / 195 / 1,206 / 153 distinct |
| Ratings | `seat_comfort`, `cabin_staff_service`, `food_and_beverages`, `inflight_entertainment`, `ground_service`, `wifi_and_connectivity`, `value_for_money` | DOUBLE | Rating 1-5 | 1.6% to 73.6% | integers 1-5 only |
| Text | `review` | VARCHAR | Free-text review | 0% | length 12-6,777 |
| Metadata | `updated_at` | TIMESTAMP | Load time | 0% | 61 values between 2026-04-03 14:30:19 and 2026-04-04 14:04:45 (loaded in batches) |

**Caveat checks:**
- Duplicate `review_id`: 0. Orphaned `airline_id`: 0. Full-row duplicates: 0.
- Leading/trailing whitespace: 0 in every text column.
- No future dates and no placeholder dates before 1990.
- Ratings hold whole numbers 1-5 only. `type_of_traveller` and `seat_type` have no spelling or case variants.
- **`date_flown` later than `date_submitted`: 41 rows** (e.g. review 49636 submitted 2016-01-11, flown 2016-12-01). `date_flown` is the first day of a month, but these gaps are still several months, so the date is wrong or mistyped.
- **Route columns:** `origin = destination` in 35 rows by city and 20 by airport. 40,387 rows have a city but no airport code, so airport columns are much emptier than city columns. `aircraft` is free text with many near-duplicates (`Boeing 737`, `Boeing 737-800`).
- **`nationality` is polluted with free text.** 238 rows are longer than 25 characters, 176 contain a digit or a dot (e.g. `B747`, `A320`, `flt. 93`, `7:30AM and 8:35PM flight`), 9 are two characters or fewer. Text from the review leaked into the wrong column.
- **`customer_name` ends with a number in 150,886 of 156,323 rows** (e.g. `Joshua Torres 6`). The suffix is not part of a real name. 1 null name (`review_id` 141473).
- **Duplicate `review` text:** 13,958 rows share their text with another row (6,972 groups, up to 5 copies); 261 groups are the same airline with the same text, which are true duplicate reviews (for example `review_id` 1 and 2).
- **Review text edge cases:** 6 rows contain only a marker (`Not Verified` 5 times, `Trip Verified` (prefixed by a check-mark character in the data) once).
- High nulls in `transit_*`, `aircraft`, `wifi_and_connectivity`, the airport codes and `inflight_entertainment` are real missing data (many reviewers do not fill them).

**Conclusion:** Keys and the join to `raw_airlines` are sound. In staging: clean `nationality` (keep only values that match a country), strip the numeric suffix from `customer_name`, fix or null the 41 `date_flown` rows, standardise `aircraft`, decide how to treat duplicate review text (at least the 261 same-airline duplicates), and drop or keep `updated_at` with a note. Do not fill the nulls.